# ModernBERT-base verifier — Kaggle version (final model: v3b)

Final model: **ModernBERT v3b** (test AUROC 0.771, D-029). ModernBERT-base on **one T4** (`USE_BOTH_GPUS = False`; the first run on 2 x T4 with `nn.DataParallel` took 7.75 hours, D-020). Settings of the v3b run: `DATASET = "final"`, `REAL_WEIGHT = 3`, `NORMALIZE_AGENT_TEXT = False`, `REAL_CV = False`, policy block on, 4,096 tokens, 3 epochs, lr 2e-5, effective batch 16; about 1.9 hours. Set `BENCHMARK_STEPS = 100` first for a timing run.

**Run it on Kaggle** (unattended, "Save & Run All (Commit)"; the exact steps are in `docs/kaggle_runbook.md`):
1. Accelerator **GPU T4**, **Internet on** (the base model is downloaded from Hugging Face), and add the private dataset **`toolcall-verifier-data-v3b`** (`final_train.jsonl`, `final_val.jsonl`, `final_test.jsonl`, `final_test_synth.jsonl`: 7,154 / 749 / 363 / 374 rows) as the only input. The data folder is found by searching `/kaggle/input/` for the file names.
2. Everything is written to `/kaggle/working/`: the model in `checkpoints/phase5_v2/best/`, `eval_report.txt`, `test_predictions.jsonl`, `test_synth_predictions.jsonl`, `val_predictions.jsonl` and `train_info.json` (legacy mode: `test_real_predictions.jsonl`).
3. **Continue a stopped run:** add the earlier version's output as an input of the new run. Its Trainer checkpoints are copied in and training resumes; a finished model there makes the notebook skip straight to evaluation.
4. **Evaluate only:** set `EVAL_ONLY = True`; with `EVAL_CHECKPOINT = None` the finished `best/` of an attached notebook output is found automatically, or give the folder in `EVAL_CHECKPOINT`.
5. Training stops, saves and evaluates when `TIME_BUDGET_HOURS` (11 h) of the 12-hour session have passed.
6. `DATASET = "final"` reads the four files, takes the split from each row, and ends with the final report (AUROC with a 95% interval, recall at 5% and 10% false alarms, per class by origin, per domain, the synthetic test set, length and source baselines). `"legacy"` is the first-round 7,580-row setup with `test_real.jsonl`.


---


In [ ]:
# ================= run configuration (the only cell you normally edit) =================
DATASET = "final"          # "final": the builder's output (files final_train/val/test/test_synth.jsonl in the data dataset), split taken from the rows,
                           #          INCLUDE_POLICY on (ModernBERT). "legacy": the 7,580-row dataset of D-012 with the frozen split (needs the 7 old files).
EVAL_ONLY = False          # True: skip training and evaluate EVAL_CHECKPOINT (needs no GPU; slow on CPU)
EVAL_CHECKPOINT = None     # folder of a saved model: ".../best", or a Trainer ".../trainer/checkpoint-N" folder. None = this run's best/
TIME_BUDGET_HOURS = 11.0   # measured from the start of this notebook; training stops, saves a checkpoint and goes on to evaluation
INCLUDE_POLICY_OVERRIDE = None  # None: the POLICY block (ModernBERT) follows DATASET ("final": on). False = off, e.g. to score a model that was trained without it
USE_BOTH_GPUS = False      # False (default): train on GPU 0 only. True: nn.DataParallel over both T4s; the first Kaggle run (2 x T4, DataParallel) took 7.75 h, see D-020. Use BENCHMARK_STEPS to compare
BENCHMARK_STEPS = None     # e.g. 100: train that many optimizer steps, print seconds per step and the projected time of a full run, then stop WITHOUT saving (about 15 minutes)
GRADIENT_CHECKPOINTING = None  # None: try without it first (faster) and switch it on after an out-of-memory error. True / False forces it
NORMALIZE_AGENT_TEXT = False  # True: every assistant text turn is unwrapped from JSON ({ "response": "..." }, the style of the gpt-5-nano agent of the induced runs) and cut to AGENT_TEXT_CHARS characters in EVERY split before the input is built, removing visible signatures of induced rows (D-027). False = as in final_v1 / final_v2
AGENT_TEXT_CHARS = 200        # the cut used when NORMALIZE_AGENT_TEXT is True
REAL_WEIGHT = 1            # integer >= 1. k: the real-origin training rows (real errors and their paired correct rows, field real_origin) appear k times per epoch, the rest once (D-026). 1 = off. The planned next run uses 3
REAL_CV = False            # True: after the evaluation, fine-tune the checkpoint on the human-reviewed real rows with grouped cross-validation and report out-of-fold scores (D-025; about 30 minutes)
REAL_CV_FOLDS = 5          # folds of whole (domain, task_id) groups over the 363 test rows
REAL_CV_EPOCHS = 3         # fine-tuning epochs per fold, on the rows of the other folds (about 290 rows)
REAL_CV_LR = 1e-5          # fine-tuning learning rate (the main run used 2e-5)
MODEL_PATH_OVERRIDE = None # folder with the base model, if Internet is off and the model is attached as an input
# ========================================================================================

import os, sys, time, json, shutil, subprocess
from pathlib import Path

if not USE_BOTH_GPUS:  # hide the second GPU before CUDA starts, so nothing can wrap the model in nn.DataParallel
    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]

T0 = time.time()  # start of the notebook; the 12-hour Kaggle limit counts from here


def detect_env():
    if os.environ.get("TCV_ENV"):  # lets a laptop test pretend to be Kaggle
        return os.environ["TCV_ENV"]
    if os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return "local"


ENV = detect_env()
KAGGLE_INPUT = Path(os.environ.get("TCV_KAGGLE_INPUT", "/kaggle/input"))
KAGGLE_WORKING = Path(os.environ.get("TCV_KAGGLE_WORKING", "/kaggle/working"))
print("environment:", ENV)


def pip_install(*packages):
    """Non-interactive pip. A failure (for example Internet switched off) is reported, not fatal: the preinstalled packages are used."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *packages],
                       capture_output=True, text=True, stdin=subprocess.DEVNULL)
    print("pip install:", "ok" if r.returncode == 0 else "FAILED, continuing with the preinstalled packages\n" + r.stderr[-500:])


def find_data_dir(name="train_pool.jsonl", max_depth=6):
    """Folder under /kaggle/input that holds `name` (Kaggle may rename the dataset folder)."""
    base_depth = len(KAGGLE_INPUT.parts)
    for dirpath, dirnames, filenames in os.walk(KAGGLE_INPUT, followlinks=True):
        if len(Path(dirpath).parts) - base_depth >= max_depth:
            dirnames[:] = []
        if name in filenames:
            return Path(dirpath)
    raise FileNotFoundError(f"{name} not found under {KAGGLE_INPUT}: attach the toolcall-verifier-data dataset to this notebook")


def fmt_td(seconds):
    s = int(max(seconds, 0))
    return f"{s // 3600}h{(s % 3600) // 60:02d}m{s % 60:02d}s"


def restore_prior_run(ckpt_dir, copy_trainers=True):
    """Kaggle: continue from the output of an earlier run that is attached as an input (Add Input > Your Work > Notebooks; its working folder shows up under
    /kaggle/input/.../<notebook>/checkpoints/<phase>/). Copies that run's Trainer checkpoints into ckpt_dir/trainer (so the Trainer resumes; not with
    copy_trainers=False, which only looks for a finished model) and returns the folder of a finished model (`best/`) if it has one."""
    if ENV != "kaggle" or not KAGGLE_INPUT.exists():
        return None
    name, finished, trainers, all_finished = ckpt_dir.name, None, {}, []
    for dirpath, dirnames, _ in os.walk(KAGGLE_INPUT, followlinks=True):
        p = Path(dirpath)
        if p.name == "best" and p.parent.name == name:
            finished = p
            all_finished.append(p)
        elif p.name.startswith("checkpoint-") and p.parent.name == "trainer" and p.parent.parent.name == name \
                and (p / "trainer_state.json").exists():
            trainers.setdefault(p.parent, []).append(p)
    best_dir = None
    if len(all_finished) > 1:  # several earlier runs attached: the most recently written one is used, and all are listed
        finished = max(all_finished, key=lambda q: (q / "adapter_config.json").stat().st_mtime if (q / "adapter_config.json").exists() else 0)
        print("several finished models are attached as inputs:", [str(q) for q in all_finished], "| using", finished)
    if finished is not None:
        info = finished / "train_info.json"
        complete = (finished / "config.json").exists() or (finished / "adapter_config.json").exists()
        if info.exists():
            complete = complete and json.load(open(info)).get("completed", True)
        if complete:
            best_dir = finished
            print("found a finished model from an earlier run:", finished)
    if trainers and best_dir is None and copy_trainers:
        def top_step(ps):
            return max(json.load(open(c / "trainer_state.json")).get("global_step", 0) for c in ps)
        parent, ps = max(trainers.items(), key=lambda kv: top_step(kv[1]))
        (ckpt_dir / "trainer").mkdir(parents=True, exist_ok=True)
        for c in ps:  # every kept checkpoint, so the best one the Trainer remembers exists too
            if not (ckpt_dir / "trainer" / c.name).exists():
                shutil.copytree(c, ckpt_dir / "trainer" / c.name)
        print("restored", sorted(c.name for c in ps), "from", parent, "to continue training")
    return best_dir


class Tee:
    """Print to the notebook and write the same text to a file, line by line, so a crash keeps what was printed.
    A complete file-like wrapper (D-029): libraries read `sys.stdout` for more than `write`; transformers' loading report calls `sys.stdout.isatty()` (and crashed the first
    Qwen evaluation with 'Tee object has no attribute isatty'). `isatty()` is False so that no colour codes reach the report file; everything else comes from the real stdout."""

    def __init__(self, path):
        self.path, self.stdout = Path(path), sys.stdout
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(self.path, "w")
        sys.stdout = self

    def write(self, s):
        self.stdout.write(s)
        self.file.write(s)
        self.file.flush()
        return len(s)

    def writelines(self, lines):
        for line in lines:
            self.write(line)

    def flush(self):
        self.stdout.flush()
        if not self.file.closed:  # a logging handler that still holds the Tee after close() must not raise at interpreter exit
            self.file.flush()

    def isatty(self):
        return False

    def writable(self):
        return True

    def readable(self):
        return False

    def seekable(self):
        return False

    def fileno(self):
        return self.stdout.fileno()  # the real stdout's descriptor (raises io.UnsupportedOperation where it has none, as the real stream would)

    @property
    def encoding(self):
        return getattr(self.stdout, "encoding", None) or "utf-8"

    @property
    def errors(self):
        return getattr(self.stdout, "errors", None) or "strict"

    @property
    def closed(self):
        return self.file.closed

    def __getattr__(self, name):  # anything else (buffer, line_buffering, newlines, ...) comes from the real stdout
        if name in ("stdout", "file", "path") or name.startswith("__"):
            raise AttributeError(name)
        return getattr(self.stdout, name)

    def close(self):
        sys.stdout = self.stdout
        self.file.close()


In [ ]:
# pip_install is defined in the configuration cell above; a failure there is reported, not fatal
pip_install("transformers>=4.48.0", "accelerate>=1.0.0", "scikit-learn")


## Cell 1: load the data (legacy: the six training files; final: `final_*.jsonl`, see the configuration cell)

The three `positives_class*_controls.jsonl` files are paired positives for the synthetic negatives of Classes 2, 4 and 6 (D-009, D-012): each keeps the injected artifact but not the error, so the artifact alone cannot predict the label.

In [ ]:
import gc, json, os, random, re
from collections import Counter
from pathlib import Path

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if ENV == "kaggle":  # data from the attached dataset, everything written to /kaggle/working, no Drive
    DATA = find_data_dir("final_train.jsonl" if DATASET == "final" else "train_pool.jsonl")
    ROOT = KAGGLE_WORKING
    CKPT_DIR = KAGGLE_WORKING / "checkpoints" / "phase5_v2"
elif ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/toolcall-verifier")
    DATA = ROOT / "data" / "derived"
    CKPT_DIR = ROOT / "checkpoints" / "phase5_v2"  # v1 checkpoints stay in checkpoints/phase5/
else:  # local checkout, or set TCV_ROOT
    ROOT = Path(os.environ.get("TCV_ROOT", "."))
    DATA = ROOT / "data" / "derived"
    CKPT_DIR = ROOT / "checkpoints" / "phase5_v2"
print("data:", DATA, "| checkpoints:", CKPT_DIR)
BEST_DIR = CKPT_DIR / "best"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SMOKE = os.environ.get("TCV_SMOKE") == "1"  # local CPU smoke test only; leave unset on Colab


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


if DATASET == "legacy":
    # file -> the label every row in it must have
    TRAIN_FILES = {
        "train_pool.jsonl": "correct",
        "bfcl_pool.jsonl": "correct",
        "positives_class2_controls.jsonl": "correct",
        "positives_class4_controls.jsonl": "correct",
        "positives_class6_controls.jsonl": "correct",
        "negatives_pool_final.jsonl": "incorrect",
    }
    rows = []
    for name, expected in TRAIN_FILES.items():
        part = load_jsonl(DATA / name)
        wrong = [r["trajectory_id"] for r in part if r["label"] != expected]
        assert not wrong, f"{name}: {len(wrong)} rows are not label={expected!r}"
        for r in part:
            r["_file"] = name
        rows += part
        print(f"{name:34s} {len(part):5d} rows  (label={expected})")

    assert len({r["trajectory_id"] for r in rows}) == len(rows), "duplicate trajectory_ids"
    EXPECTED_ROWS = 7580  # the six training files; test_real.jsonl is evaluation-only and not counted
    assert len(rows) == EXPECTED_ROWS, f"expected {EXPECTED_ROWS} training rows, found {len(rows)}: copy the current data/derived/ to Drive"
else:
    assert DATASET == "final", f'DATASET must be "legacy" or "final", not {DATASET!r}'

    def final_path(name):  # data/derived/final/NAME.jsonl, or final_NAME.jsonl next to the other data files (the Kaggle upload); TCV_FINAL_DIR overrides
        for d in (os.environ.get("TCV_FINAL_DIR"), DATA / "final", DATA):
            for fn in (f"{name}.jsonl", f"final_{name}.jsonl"):
                if d and (Path(d) / fn).exists():
                    return Path(d) / fn
        raise FileNotFoundError(f"final dataset file {name}.jsonl not found in {DATA}: run data/package_for_kaggle.py and upload it, or set DATASET = \"legacy\"")

    FINAL = {s: load_jsonl(final_path(s)) for s in ("train", "val", "test", "test_synth")}
    for part in FINAL.values():
        for r in part:
            r["_file"] = "train_pool.jsonl" if r.get("source") == "tau2_gold" else str(r.get("source"))
    rows = FINAL["train"] + FINAL["val"]
    test_rows = FINAL["test"]
    # the only failures: an empty split, or a synthetic row in the test split
    assert all(FINAL.values()), f"empty split(s): {[s for s, p in FINAL.items() if not p]}"
    bad = [r["trajectory_id"] for r in FINAL["test"] if r.get("origin") == "synthetic" or str(r.get("source", "")).startswith("synthetic")]
    assert not bad, f"synthetic rows in the test split: {bad[:5]}"
    ids = [r["trajectory_id"] for p in FINAL.values() for r in p]
    off = [s for s, p in FINAL.items() for r in p if r.get("split") not in (s, None)]
    print(f"final dataset from {final_path('train').parent}: {len(ids)} rows" + ("" if len(ids) == len(set(ids)) else f" (WARNING: {len(ids) - len(set(ids))} duplicate ids)")
          + ("" if not off else f" (WARNING: {len(off)} rows whose split field differs from their file)"))

    def final_counts(part_by_split):
        def cls(r):
            return f"class {r['error_class']}" if r["label"] == "incorrect" else "correct"
        for title, key in (("class", cls), ("source", lambda r: r.get("source")), ("origin", lambda r: r.get("origin")), ("domain", lambda r: r["domain"])):
            cnt = Counter((str(key(r)), s) for s, p in part_by_split.items() for r in p)
            names = sorted({n for n, _ in cnt})
            print(f"\nrows per {title} and split:")
            print(f"   {'':30s}" + "".join(f"{s:>11s}" for s in part_by_split) + f"{'total':>9s}")
            for n in names:
                vals = [cnt[(n, s)] for s in part_by_split]
                print(f"   {n:30s}" + "".join(f"{v:>11d}" for v in vals) + f"{sum(vals):>9d}")

    final_counts(FINAL)

LABEL2ID = {"correct": 0, "incorrect": 1}  # "incorrect" is the positive class everywhere
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print(f"\ntotal: {len(rows)} examples")
print("label balance:", dict(Counter(r["label"] for r in rows)))
print("incorrect by error class:")
for (c, n), k in sorted({(r["error_class"], r["error_class_name"]): 0 for r in rows if r["label"] == "incorrect"}.items()):
    print(f"  class {c} ({n}): {sum(1 for r in rows if r['label'] == 'incorrect' and r['error_class'] == c)}")
print("domains:", dict(Counter(r["domain"] for r in rows)))

print("\n3-row sample:")
for r in random.Random(SEED).sample(rows, 3):
    call = json.dumps(r["candidate_tool_call"], ensure_ascii=False)
    print(f"  {r['trajectory_id'][:70]}")
    print(f"    domain={r['domain']} label={r['label']} error_class={r.get('error_class')} turns={len(r['turns'])} file={r['_file']}")
    print(f"    candidate: {call[:140]}")

## Cell 2: state compression, v1 (`format_input`; kept unchanged for the comparison, training uses v2 in Cell 2b)

Target is at most 1,500 tokens (model limit 8,192; `max_length=1536` leaves a margin).

1. **Tools:** name + parameter names only (no descriptions or types), for the tools called in the last 5 turns plus the candidate's tool. If the schema text it is given contains a `# Requires: ...` line for the candidate's tool (inference time only: `data/tool_schemas.py` renders these with `include_requires=True`; the stored training and test data do not contain them, see D-009), that rule is shown beside the tool.
2. **Turns:** last 10 turns, tool results cut to 200 chars. Format `USER:` / `AGENT:` / `AGENT_CALL:` / `TOOL_RESULT:`.
3. **Candidate:** appended last as `CANDIDATE: name(k=v, ...)`.

Two additions beyond that spec, both measured and explained in docs/decision_log.md D-011:
- **`EARLIER_CALLS:`** a one-line list of tool calls that fall *before* the 10-turn window. Without it, 34% of the synthetic Class 4 (repeated call) negatives hide the earlier identical call and become indistinguishable from their positives.
- **Budget guard:** if an example is still over 1,500 tokens, the window and clip lengths shrink step by step. The candidate is the last thing in the text, so plain right-truncation would cut it off first; this guard prevents that.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "answerdotai/ModernBERT-base"
if MODEL_PATH_OVERRIDE:
    MODEL_NAME = str(MODEL_PATH_OVERRIDE)
MAX_LEN = 1536
TOKEN_BUDGET = 1500
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# tightening ladder: first level is the spec; later levels only fire when over budget
LEVELS = [
    dict(n_turns=10, tool_chars=200, msg_chars=500, earlier=20, arg_chars=100),
    dict(n_turns=10, tool_chars=100, msg_chars=300, earlier=10, arg_chars=60),
    dict(n_turns=8, tool_chars=60, msg_chars=200, earlier=5, arg_chars=40),
    dict(n_turns=6, tool_chars=40, msg_chars=150, earlier=3, arg_chars=30),
]
CAND_ARG_CHARS = 300  # candidate args are shown almost in full: the error is often in one value


def _clip(s, n):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if len(s) <= n else s[: n - 1] + "…"


def _val(v, n):
    return _clip(v if isinstance(v, str) else json.dumps(v, ensure_ascii=False), n)


def _call(name, args, n):
    return f"{name}(" + ", ".join(f"{k}={_val(v, n)}" for k, v in (args or {}).items()) + ")"


def _split_top(s):
    parts, depth, cur = [], 0, ""
    for ch in s:
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth -= 1
        if ch == "," and depth == 0:
            parts.append(cur)
            cur = ""
        else:
            cur += ch
    if cur.strip():
        parts.append(cur)
    return parts


_schema_cache = {}
_requires_cache = {}


def parse_requires(text):
    # {tool_name: "prerequisite rule"} from the "  # Requires: ..." line that follows a tool in
    # the tau2-bench schema text (data/tool_schemas.py PREREQUISITES). BFCL schemas have none.
    if text in _requires_cache:
        return _requires_cache[text]
    rules, cur = {}, None
    for line in text.splitlines():
        if line.startswith("- ") and "(" in line:
            cur = line[2 : line.index("(")].strip()
        elif cur and line.startswith("  # Requires:"):
            rules[cur] = line[len("  # Requires:"):].strip()
    _requires_cache[text] = rules
    return rules



def parse_schema(text):
    # {tool_name: [param names]} from either schema text format in the data:
    #   tau2-bench: "- name(p: type, q: type (optional)): description"
    #   BFCL:       "Function: name" / "  Parameters:" / "    p (type) [required]: description"
    if text in _schema_cache:
        return _schema_cache[text]
    tools, cur = {}, None
    for line in text.splitlines():
        if line.startswith("- ") and "(" in line:
            name = line[2 : line.index("(")].strip()
            depth, end = 0, None
            for i in range(line.index("("), len(line)):
                depth += (line[i] == "(") - (line[i] == ")")
                if depth == 0:
                    end = i
                    break
            inner = line[line.index("(") + 1 : end] if end else ""
            tools[name] = [p.split(":")[0].strip() for p in _split_top(inner) if p.strip()]
        elif line.startswith("Function: "):
            cur = line[len("Function: "):].strip()
            tools[cur] = []
        elif cur and re.match(r"^    \w+ \(", line):
            tools[cur].append(line.split("(")[0].strip())
    _schema_cache[text] = tools
    return tools


def _format(ex, n_turns, tool_chars, msg_chars, earlier, arg_chars):
    turns = ex["turns"]
    cand = ex["candidate_tool_call"]
    start = max(0, len(turns) - n_turns)
    window = turns[start:]

    # 1. tools: called in the last 5 turns, plus the candidate's tool
    schema = parse_schema(ex.get("tool_schemas_text", ""))
    wanted = []
    for t in turns[-5:]:
        if t.get("role") == "assistant_tool_call" and t["name"] not in wanted:
            wanted.append(t["name"])
    if cand["name"] not in wanted:
        wanted.append(cand["name"])
    tool_strs = [f"{n}({', '.join(schema[n])})" for n in wanted if n in schema]
    # the prerequisite rule is shown for the candidate's tool only: it is what the call is judged
    # against, and rules for every listed tool would cost tokens
    rule = parse_requires(ex.get("tool_schemas_text", "")).get(cand["name"])
    if rule:
        lines = ["TOOLS:"] + [f"  {s}" + (f"  # Requires: {rule}" if s.startswith(cand["name"] + "(") else "") for s in tool_strs]
    else:
        lines = ["TOOLS: " + ("; ".join(tool_strs) if tool_strs else "(none listed)")]

    # 2. calls that fall before the window (one line, most recent last)
    before = [t for t in turns[:start] if t.get("role") == "assistant_tool_call"]
    if before and earlier:
        lines.append("EARLIER_CALLS: " + "; ".join(_call(t["name"], t["arguments"], arg_chars) for t in before[-earlier:]))

    # 3. the window
    lines.append("HISTORY:")
    for t in window:
        role = t.get("role")
        if role == "user":
            lines.append("USER: " + _clip(t.get("content", ""), msg_chars))
        elif role == "assistant":
            lines.append("AGENT: " + _clip(t.get("content", ""), msg_chars))
        elif role == "assistant_tool_call":
            lines.append("AGENT_CALL: " + _call(t["name"], t["arguments"], arg_chars))
        elif role == "tool":
            lines.append("TOOL_RESULT: " + _clip(t.get("content", ""), tool_chars))
        else:
            lines.append(f"{str(role).upper()}: " + _clip(t.get("content", ""), msg_chars))

    # 4. candidate, always last
    lines.append("CANDIDATE: " + _call(cand["name"], cand["arguments"], CAND_ARG_CHARS))
    return "\n".join(lines)


def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def format_input(example, tok=True):
    # Returns the model input string. With tok=True the text is tightened until it fits TOKEN_BUDGET.
    for lvl in LEVELS:
        text = _format(example, **lvl)
        if not tok or n_tokens(text) <= TOKEN_BUDGET:
            return text
    return text


# --- token length distribution over 200 random training examples ---
sample = random.Random(SEED).sample(rows, 200)
natural = np.array([n_tokens(_format(r, **LEVELS[0])) for r in sample])
final = np.array([n_tokens(format_input(r)) for r in sample])
pct = lambda a: {f"p{p}": int(np.percentile(a, p)) for p in (50, 90, 99)} | {"max": int(a.max())}
print("tokens, spec settings only :", pct(natural), f"| over {TOKEN_BUDGET}: {(natural > TOKEN_BUDGET).sum()}/200")
print("tokens, after budget guard :", pct(final), f"| over {TOKEN_BUDGET}: {(final > TOKEN_BUDGET).sum()}/200")

# --- does the evidence survive compression? Class 4 needs the earlier identical call to be visible ---
def dup_visible(r, earlier):
    # is the candidate call (formatted exactly as in the history) present before the CANDIDATE line?
    lvl = {**LEVELS[0], "earlier": earlier}
    cs = _call(r["candidate_tool_call"]["name"], r["candidate_tool_call"]["arguments"], lvl["arg_chars"])
    return cs in _format(r, **lvl).split("CANDIDATE:")[0]

c4 = [r for r in rows if r["label"] == "incorrect" and r["error_class"] == 4]
print(f"class 4 negatives with the earlier duplicate visible: last-10 window only {sum(dup_visible(r, 0) for r in c4)}/{len(c4)}"
      f"  ->  with EARLIER_CALLS {sum(dup_visible(r, LEVELS[0]['earlier']) for r in c4)}/{len(c4)}")

ex = next(r for r in c4 if len(r["turns"]) > 14)
print("\n--- one formatted example (class 4 negative) ---")
print(format_input(ex)[:3000])

## Cell 2b: evidence-aware input (`format_input_v2`) and the v1 vs v2 measurement

Rules (D-017), on top of v1's `TOOLS:` / `EARLIER_CALLS:` / `HISTORY:` / `CANDIDATE:` layout:
- keep up to the last **30** turns; calls before the window are summarised in `EARLIER_CALLS` as in v1;
- a tool result is shown **in full** if it contains any string value (3+ characters, each list item separately) from the candidate call's arguments, and the **2 most recent** tool results are always shown in full;
- every other tool result is clipped to 200 characters, user and agent messages to 500;
- `CANDIDATE:` is always last. Budget 4,096 tokens; if a row is over, tighten in this order: older non-evidence results, then messages, then turns kept. Full-evidence results and the candidate are never clipped.

The next cell measures v1 vs v2: token percentiles, rows that hit the budget, the share of tool-result characters visible, and whether the earlier duplicate call of the Class 4 negatives is still visible. It takes about a minute.


In [ ]:
# --- v2: evidence-aware input (D-017). v1 above is left untouched and is still used by the Phase 5 v1 numbers. ---
MAX_LEN = 4096          # v2 tokenization / training length (v1 used 1536); the v1 functions above keep using TOKEN_BUDGET = 1500
TOKEN_BUDGET_V2 = 4096
# --- policy snippets (begin) ---
# INCLUDE_POLICY adds a `POLICY:` block with the policy text of the candidate's tool (D-021). On in DATASET = "final" mode (D-023), off in legacy.
# The text depends on the tool only (never on the row), copied from the benchmark's policy documents by data/policy_snippets.py.
INCLUDE_POLICY = (DATASET == "final") if INCLUDE_POLICY_OVERRIDE is None else bool(INCLUDE_POLICY_OVERRIDE)
print("INCLUDE_POLICY =", INCLUDE_POLICY)
POLICY_SNIPPETS = {
    "retail::cancel_pending_order": "An order can only be cancelled if its status is 'pending', and you should check its status before taking the action. The user needs to confirm the order id and the reason (either 'no longer needed' or 'ordered by mistake') for cancellation. Other reasons are not acceptable. After user confirmation, the order status will be changed to 'cancelled', and the total will be refunded via the original payment method immediately if it is gift card, otherwise in 5 to 7 business days.",
    "retail::modify_pending_order_address": "An order can only be modified if its status is 'pending', and you should check its status before taking the action. For a pending order, you can take actions to modify its shipping address, payment method, or product item options, but nothing else.",
    "retail::modify_pending_order_payment": "An order can only be modified if its status is 'pending', and you should check its status before taking the action. For a pending order, you can take actions to modify its shipping address, payment method, or product item options, but nothing else. The user can only choose a single payment method different from the original payment method. If the user wants the modify the payment method to gift card, it must have enough balance to cover the total amount. After user confirmation, the order status will be kept as 'pending'. The original payment method will be refunded immediately if it is a gift card, otherwise it will be refunded within 5 to 7 business days.",
    "retail::modify_pending_order_items": "An order can only be modified if its status is 'pending', and you should check its status before taking the action. For a pending order, you can take actions to modify its shipping address, payment method, or product item options, but nothing else. This action can only be called once, and will change the order status to 'pending (items modifed)'. The agent will not be able to modify or cancel the order anymore. So you must confirm all the details are correct and be cautious before taking this action. In particular, remember to remind the customer to confirm they have provided all the items they want to modify. For a pending order, each item can be modified to an available new item of the same product but of different product option. There cannot be any change of product types, e.g. modify shirt to shoe. The user must provide a payment method to pay or receive refund of the price difference. If the user provides a gift card, it must have enough balance to cover the price difference.",
    "retail::return_delivered_order_items": "An order can only be returned if its status is 'delivered', and you should check its status before taking the action. The user needs to confirm the order id and the list of items to be returned. The user needs to provide a payment method to receive the refund. The refund must either go to the original payment method, or an existing gift card. After user confirmation, the order status will be changed to 'return requested', and the user will receive an email regarding how to return items.",
    "retail::exchange_delivered_order_items": "An order can only be exchanged if its status is 'delivered', and you should check its status before taking the action. In particular, remember to remind the customer to confirm they have provided all items to be exchanged. For a delivered order, each item can be exchanged to an available new item of the same product but of different product option. There cannot be any change of product types, e.g. modify shirt to shoe. The user must provide a payment method to pay or receive refund of the price difference. If the user provides a gift card, it must have enough balance to cover the price difference. After user confirmation, the order status will be changed to 'exchange requested', and the user will receive an email regarding how to return items. There is no need to place a new order.",
    "airline::book_reservation": "The agent must first obtain the user id from the user. The agent should then ask for the trip type, origin, destination. Cabin: - Cabin class must be the same across all the flights in a reservation. Passengers: - Each reservation can have at most five passengers. - The agent needs to collect the first name, last name, and date of birth for each passenger. - All passengers must fly the same flights in the same cabin. Payment: - Each reservation can use at most one travel certificate, at most one credit card, and at most three gift cards. - The remaining amount of a travel certificate is not refundable. - All payment methods must already be in user profile for safety reasons. Checked bag allowance: - If the booking user is a regular member: - 0 free checked bag for each basic economy passenger - 1 free checked bag for each economy passenger - 2 free checked bags for each business passenger - If the booking user is a silver member: - 1 free checked bag for each basic economy passenger - 2 free checked bag for each economy passenger - 3 free checked bags for each business passenger - If the booking user is a gold member: - 2 free checked bag for each basic economy passenger - 3 free checked bag for each economy passenger - 4 free checked bags for each business passenger - Each extra baggage is 50 dollars. Do not add checked bags that the user does not need. Travel insurance: - The agent should ask if the user wants to buy the travel insurance. - The travel insurance is 30 dollars per passenger and enables full refund if the user needs to cancel the flight given health or weather reasons.",
    "airline::update_reservation_flights": "Change flights: - Basic economy flights cannot be modified. - Other reservations can be modified without changing the origin, destination, and trip type. - Some flight segments can be kept, but their prices will not be updated based on the current price. - The API does not check these for the agent, so the agent must make sure the rules apply before calling the API! Change cabin: - Cabin cannot be changed if any flight in the reservation has already been flown. - In other cases, all reservations, including basic economy, can change cabin without changing the flights. - Cabin class must remain the same across all the flights in the same reservation; changing cabin for just one flight segment is not possible. - If the price after cabin change is higher than the original price, the user is required to pay for the difference. - If the price after cabin change is lower than the original price, the user is should be refunded the difference. Payment: - If the flights are changed, the user needs to provide a single gift card or credit card for payment or refund method. The payment method must already be in user profile for safety reasons.",
    "airline::update_reservation_baggages": "Change baggage and insurance: - The user can add but not remove checked bags. - The user cannot add insurance after initial booking.",
    "airline::update_reservation_passengers": "Change passengers: - The user can modify passengers but cannot modify the number of passengers. - Even a human agent cannot modify the number of passengers.",
    "airline::cancel_reservation": "First, the agent must obtain the user id and reservation id. - The user must provide their user id. - If the user doesn't know their reservation id, the agent should help locate it using available tools. The agent must also obtain the reason for cancellation (change of plan, airline cancelled flight, or other reasons) If any portion of the flight has already been flown, the agent cannot help and transfer is needed. Otherwise, flight can be cancelled if any of the following is true: - The booking was made within the last 24 hrs - The flight is cancelled by airline - It is a business flight - The user has travel insurance and the reason for cancellation is covered by insurance. The API does not check that cancellation rules are met, so the agent must make sure the rules apply before calling the API! Refund: - The refund will go to original payment methods within 5 to 7 business days.",
    "airline::send_certificate": "Do not proactively offer a compensation unless the user explicitly asks for one. Do not compensate if the user is regular member and has no travel insurance and flies (basic) economy. Always confirms the facts before offering compensation. Only compensate if the user is a silver/gold member or has travel insurance or flies business. - If the user complains about cancelled flights in a reservation, the agent can offer a certificate as a gesture after confirming the facts, with the amount being $100 times the number of passengers. - If the user complains about delayed flights in a reservation and wants to change or cancel the reservation, the agent can offer a certificate as a gesture after confirming the facts and changing or cancelling the reservation, with the amount being $50 times the number of passengers. Do not offer compensation for any other reason than the ones listed above.",
    "telecom::resume_line": "When a line is suspended, the user will not have service. A line can be suspended for the following reasons: - The user has an overdue bill. - The line's contract end date is in the past. You are allowed to lift the suspension after the user has paid all their overdue bills. You are not allowed to lift the suspension if the line's contract end date is in the past, even if the user has paid all their overdue bills. After you resume the line, the user will have to reboot their device to get service.",
    "telecom::suspend_line": "When a line is suspended, the user will not have service. A line can be suspended for the following reasons: - The user has an overdue bill. - The line's contract end date is in the past. You are allowed to lift the suspension after the user has paid all their overdue bills. You are not allowed to lift the suspension if the line's contract end date is in the past, even if the user has paid all their overdue bills. After you resume the line, the user will have to reboot their device to get service.",
    "telecom::send_payment_request": "You can help the user make a payment for an overdue bill. To do so you need to follow these steps: - Check the bill status to make sure it is overdue. - Check the bill amount due - Send the user a payment request for the overdue bill. - This will change the status of the bill to AWAITING PAYMENT. - Inform the user that a payment request has been sent. They should: - Check their payment requests using the check_payment_request tool. - If the user accepts the payment request, use the make_payment tool to make the payment. - After the payment is made, the bill status will be updated to PAID. - Always check that the bill status is updated to PAID before informing the user that the bill has been paid. Important: - A user can only have one bill in the AWAITING PAYMENT status at a time. - The send payement request tool will not check if the bill is overdue. You should always check that the bill is overdue before sending a payment request.",
    "telecom::refuel_data": "Each plan specify the maxium data usage per month. If the user's data usage for a line exceeds the plan's data limit, data connectivity will be lost. You can add more data to the line by \"refueling\" data at a price per GB specified by the plan. The maximum amount of data that can be refueled is 2GB. To refuel data you should: - Ask them how much data they want to refuel - Confirm the price - Apply the refueled data to the line associated with the phone number the user provided.",
    "telecom::enable_roaming": "If a line is roaming enabled, the user can use their phone's data connection in areas outside their home network. We offer data roaming to users who are traveling outside their home network. If a user is traveling outside their home network, you should check if the line is roaming enabled. If it is not, you should enable it at no cost for the user.",
    "telecom::disable_roaming": "If a line is roaming enabled, the user can use their phone's data connection in areas outside their home network. We offer data roaming to users who are traveling outside their home network. If a user is traveling outside their home network, you should check if the line is roaming enabled. If it is not, you should enable it at no cost for the user.",
}
# --- policy snippets (end) ---

# Tightening ladder, applied only while a row is over TOKEN_BUDGET_V2, in this order:
#   1. older non-evidence tool results (200 -> 100 -> 50 -> 20 chars)
#   2. user / agent messages (500 -> 300 -> 200 -> 120 chars)
#   3. turns kept (30 -> 20 -> 12 -> 8 -> 5 -> 3)
# Never clipped at any level: tool results that contain a candidate argument value, the 2 most recent
# tool results, and the candidate (args shown up to CAND_ARG_CHARS, as in v1).
LEVELS_V2 = (
    [dict(n_turns=30, tool_chars=t, msg_chars=500, earlier=20, arg_chars=100) for t in (200, 100, 50, 20)]
    + [dict(n_turns=30, tool_chars=20, msg_chars=m, earlier=20, arg_chars=100) for m in (300, 200, 120)]
    + [dict(n_turns=n, tool_chars=20, msg_chars=120, earlier=20, arg_chars=100) for n in (20, 12, 8, 5, 3)]
)
MIN_NEEDLE = 3  # candidate string values shorter than this are ignored when looking for evidence


def _needles(args):
    # every string value of the candidate call's arguments (3+ chars); list items and nested values are checked separately
    out = []

    def walk(v):
        if isinstance(v, str):
            if len(v) >= MIN_NEEDLE:
                out.append(v)
        elif isinstance(v, (list, tuple)):
            for x in v:
                walk(x)
        elif isinstance(v, dict):
            for x in v.values():
                walk(x)

    walk(args or {})
    return out


def _norm(s):
    return re.sub(r"\s+", " ", str(s)).strip()


def _format_v2(ex, n_turns, tool_chars, msg_chars, earlier, arg_chars):
    # returns (text, shown_chars, total_chars): tool-result characters shown vs present in the whole conversation
    turns = ex["turns"]
    cand = ex["candidate_tool_call"]
    start = max(0, len(turns) - n_turns)
    needles = _needles(cand.get("arguments"))
    tool_idx = [i for i, t in enumerate(turns) if t.get("role") == "tool"]
    recent2 = set(tool_idx[-2:])

    # TOOLS: same rule as v1 (called in the last 5 turns + the candidate's tool, parameter names only)
    schema = parse_schema(ex.get("tool_schemas_text", ""))
    wanted = []
    for t in turns[-5:]:
        if t.get("role") == "assistant_tool_call" and t["name"] not in wanted:
            wanted.append(t["name"])
    if cand["name"] not in wanted:
        wanted.append(cand["name"])
    tool_strs = [f"{n}({', '.join(schema[n])})" for n in wanted if n in schema]
    rule = parse_requires(ex.get("tool_schemas_text", "")).get(cand["name"])
    if rule:
        lines = ["TOOLS:"] + [f"  {s}" + (f"  # Requires: {rule}" if s.startswith(cand["name"] + "(") else "") for s in tool_strs]
    else:
        lines = ["TOOLS: " + ("; ".join(tool_strs) if tool_strs else "(none listed)")]

    # POLICY: the policy text of the candidate's tool (D-021), only when INCLUDE_POLICY is on; never clipped
    policy = POLICY_SNIPPETS.get(f"{ex.get('domain')}::{cand['name']}") if INCLUDE_POLICY else None
    if policy:
        lines.append("POLICY: " + policy)

    # EARLIER_CALLS: calls that fall before the window (one line, most recent last), as in v1
    before = [t for t in turns[:start] if t.get("role") == "assistant_tool_call"]
    if before and earlier:
        lines.append("EARLIER_CALLS: " + "; ".join(_call(t["name"], t["arguments"], arg_chars) for t in before[-earlier:]))

    lines.append("HISTORY:")
    shown = 0
    for i in range(start, len(turns)):
        t = turns[i]
        role = t.get("role")
        if role == "user":
            lines.append("USER: " + _clip(t.get("content", ""), msg_chars))
        elif role == "assistant":
            lines.append("AGENT: " + _clip(t.get("content", ""), msg_chars))
        elif role == "assistant_tool_call":
            lines.append("AGENT_CALL: " + _call(t["name"], t["arguments"], arg_chars))
        elif role == "tool":
            raw = str(t.get("content", ""))
            full = i in recent2 or any(n in raw for n in needles)
            text = _norm(raw) if full else _clip(raw, tool_chars)
            shown += len(text) - (0 if full or not text.endswith("…") else 1)
            lines.append("TOOL_RESULT: " + text)
        else:
            lines.append(f"{str(role).upper()}: " + _clip(t.get("content", ""), msg_chars))
    lines.append("CANDIDATE: " + _call(cand["name"], cand["arguments"], CAND_ARG_CHARS))
    total = sum(len(_norm(turns[i].get("content", ""))) for i in tool_idx)
    return "\n".join(lines), shown, total


def format_input_v2(example, tok=True, meta=False):
    # Model input string (v2). With tok=True the ladder is walked until the text fits TOKEN_BUDGET_V2.
    # meta=True returns (text, dict(level, tokens, shown, total, over_budget, natural_tokens, arg_chars)).
    first = None
    for li, lvl in enumerate(LEVELS_V2):
        text, shown, total = _format_v2(example, **lvl)
        nt = n_tokens(text) if tok else None
        if first is None:
            first = nt
        if not tok or nt <= TOKEN_BUDGET_V2:
            break
    if not meta:
        return text
    return text, dict(level=li, tokens=nt, natural_tokens=first, shown=shown, total=total,
                      over_budget=bool(tok and nt > TOKEN_BUDGET_V2), arg_chars=lvl["arg_chars"])

# --- NORMALIZE_AGENT_TEXT (D-027): the wrapper around format_input_v2 (every tokenization and every evaluation calls it) ---
_format_input_v2_raw = format_input_v2


def _plain_agent_text(text):
    """The plain text of an agent message: a JSON object is unwrapped to its text value (keys response, text, message, else the longest string value), then the text is cut."""
    s = text.lstrip()
    if s.startswith("{"):
        try:
            obj = json.loads(s)
        except ValueError:
            obj = None
        if isinstance(obj, dict):
            for k in ("response", "text", "message"):
                if isinstance(obj.get(k), str):
                    text = obj[k]
                    break
            else:
                vals = [v for v in obj.values() if isinstance(v, str)]
                if vals:
                    text = max(vals, key=len)
    return text if len(text) <= AGENT_TEXT_CHARS else text[: AGENT_TEXT_CHARS - 1] + "…"


def _normalized_example(example):
    turns, changed = [], False
    for t in example["turns"]:
        if t.get("role") == "assistant" and isinstance(t.get("content"), str) and t["content"]:
            new = _plain_agent_text(t["content"])
            if new != t["content"]:
                t, changed = {**t, "content": new}, True
        turns.append(t)
    return {**example, "turns": turns} if changed else example


def format_input_v2(example, tok=True, meta=False):
    return _format_input_v2_raw(_normalized_example(example) if NORMALIZE_AGENT_TEXT else example, tok=tok, meta=meta)


print(f"NORMALIZE_AGENT_TEXT = {NORMALIZE_AGENT_TEXT}" + (f": assistant text turns are unwrapped from JSON and cut to {AGENT_TEXT_CHARS} characters in every split" if NORMALIZE_AGENT_TEXT else ""))


In [ ]:
# --- v1 vs v2: token lengths and how much evidence survives (train pool + test_real) ---
test_rows = load_jsonl(DATA / "test_real.jsonl") if DATASET == "legacy" else FINAL["test"]
train_pool = [r for r in rows if r["_file"] == "train_pool.jsonl"]  # the 1,185 original tau2-bench trajectories


def v1_meta(ex):
    for li, lvl in enumerate(LEVELS):
        text = _format(ex, **lvl)
        nt = n_tokens(text)
        if nt <= TOKEN_BUDGET:
            break
    tool = [t for t in ex["turns"][max(0, len(ex["turns"]) - lvl["n_turns"]):] if t.get("role") == "tool"]
    shown = sum(len(_norm(t.get("content", ""))) if len(_norm(t.get("content", ""))) <= lvl["tool_chars"] else lvl["tool_chars"] - 1 for t in tool)
    total = sum(len(_norm(t.get("content", ""))) for t in ex["turns"] if t.get("role") == "tool")
    return text, dict(level=li, tokens=nt, shown=shown, total=total, over_budget=nt > TOKEN_BUDGET, arg_chars=lvl["arg_chars"])


def full_tokens(ex, schema=False):
    # the whole conversation, nothing clipped, plus the candidate (optionally with the full tool schema text)
    parts = [ex["tool_schemas_text"]] if schema else []
    for t in ex["turns"]:
        parts.append(f"{t.get('role')}: " + (json.dumps({"name": t["name"], "arguments": t["arguments"]}) if t.get("role") == "assistant_tool_call" else str(t.get("content", ""))))
    parts.append("CANDIDATE: " + json.dumps(ex["candidate_tool_call"]))
    return n_tokens("\n".join(parts))


def summarize(part):
    out = {}
    for name, fn in (("v1", v1_meta), ("v2", lambda r: format_input_v2(r, meta=True))):
        ms = [fn(r) for r in part]
        out[name] = dict(ms=[m[1] for m in ms], texts=[m[0] for m in ms], toks=np.array([m[1]["tokens"] for m in ms]))
    return out


def pctl(a):
    return {f"p{p}": int(np.percentile(a, p)) for p in (50, 90, 99)} | {"max": int(np.max(a))}


def report(label, part, res):
    print(f"\n===== {label}: {len(part)} rows =====")
    for name, budget in (("v1", TOKEN_BUDGET), ("v2", TOKEN_BUDGET_V2)):
        ms = res[name]["ms"]
        toks = res[name]["toks"]
        nat = np.array([m.get("natural_tokens", m["tokens"]) for m in ms]) if name == "v2" else toks
        print(f"{name} tokens {pctl(toks)} | budget {budget}: ladder fired on {int((nat > budget).sum())} rows, still over after the ladder {int(sum(m['over_budget'] for m in ms))}")
        sh = [(m["shown"] / m["total"]) for m in ms if m["total"] > 0]
        print(f"   tool-result chars visible: median {np.median(sh):.1%} (mean {np.mean(sh):.1%}, pooled {sum(m['shown'] for m in ms) / max(sum(m['total'] for m in ms), 1):.1%}) over {len(sh)} rows that have tool results")


def class4_visible(part, res, name):
    n = v = 0
    for r, text, m in zip(part, res[name]["texts"], res[name]["ms"]):
        if r["label"] == "incorrect" and r["error_class"] == 4:
            cs = _call(r["candidate_tool_call"]["name"], r["candidate_tool_call"]["arguments"], m["arg_chars"])
            n += 1
            v += cs in text.split("CANDIDATE:")[0]
    return v, n


res_test = summarize(test_rows)
report("test_real", test_rows, res_test)
res_train = summarize(rows)
report("training rows (7,580)", rows, res_train)
res_pool = {k: dict(ms=[res_train[k]["ms"][i] for i, r in enumerate(rows) if r["_file"] == "train_pool.jsonl"],
                    toks=np.array([res_train[k]["ms"][i]["tokens"] for i, r in enumerate(rows) if r["_file"] == "train_pool.jsonl"])) for k in ("v1", "v2")}
n_fit = sum(m["natural_tokens"] <= TOKEN_BUDGET_V2 for m in res_pool["v2"]["ms"])
n_fit_final = sum(not m["over_budget"] for m in res_pool["v2"]["ms"])
print(f"\ntrain_pool.jsonl trajectories ({len(train_pool)}): v2 fits {TOKEN_BUDGET_V2} tokens before any tightening: {n_fit}; after the ladder: {n_fit_final}")
print(f"test_real: v2 fits {TOKEN_BUDGET_V2} before tightening: {sum(m['natural_tokens'] <= TOKEN_BUDGET_V2 for m in res_test['v2']['ms'])}/{len(test_rows)}; after the ladder: {sum(not m['over_budget'] for m in res_test['v2']['ms'])}/{len(test_rows)}")
ft = np.array([full_tokens(r) for r in test_rows])
fts = np.array([full_tokens(r, schema=True) for r in test_rows])
print(f"test_real: full uncompressed conversation fits 8,192 tokens: {int((ft <= 8192).sum())}/{len(test_rows)} without the tool schema text, {int((fts <= 8192).sum())}/{len(test_rows)} with it (max {int(ft.max())} / {int(fts.max())})")

print("\nClass 4 negatives whose earlier duplicate call is visible:")
for nm, res, part in (("v1", res_train, rows), ("v2", res_train, rows)):
    v, n = class4_visible(part, res, nm)
    print(f"  {nm}: {v}/{n} ({v / n:.1%})")

print("\nreal errors in test_real: tool-result chars visible (per row, v1 -> v2)")
cls_of = [r["error_class"] for r in test_rows]
for c in (1, 3, 4, 5, 7):
    idx = [i for i, r in enumerate(test_rows) if r["label"] == "incorrect" and r["error_class"] == c]
    f = lambda name: [res_test[name]["ms"][i]["shown"] / res_test[name]["ms"][i]["total"] for i in idx if res_test[name]["ms"][i]["total"] > 0]
    a, b = f("v1"), f("v2")
    if a:
        print(f"  class {c} (n={len(idx)}): v1 median {np.median(a):.0%} (range {min(a):.0%}-{max(a):.0%})  ->  v2 median {np.median(b):.0%} (range {min(b):.0%}-{max(b):.0%})")


## Cell 3: train/val split (legacy mode)

90/10, stratified on `label`, seed 42, **frozen**: the validation groups are listed in the cell below (6,816 train / 764 val rows), because the fold `StratifiedGroupKFold` returns depends on the scikit-learn version (6,892 / 688 up to 1.7, 6,816 / 764 from 1.8). Rows are **grouped by `(domain, task_id)`** so that a group never straddles train and val. This covers the required control pairing for Classes 2, 4 and 6 (a negative and its control share a task, and the `paired_negative_id` link is asserted per class below) and also keeps every negative together with the positive it was derived from, which a random row-level split would not.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

groups = [f"{r['domain']}::{r['task_id']}" for r in rows]
y = [LABEL2ID[r["label"]] for r in rows]
# --- frozen split (begin) ---
# The grouped fold StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42) returns depends on the
# scikit-learn version: 6,816 train / 764 val from 1.8 on (the split D-012 documents), 6,892 / 688 up to 1.7.
# So the validation groups are frozen here (computed with scikit-learn 1.9.1) and
# every environment trains on the same split. Every row of a (domain, task_id) group goes to the same side.
VAL_GROUPS = {
    "bfcl::multi_turn_base_108",
    "bfcl::multi_turn_base_145",
    "bfcl::multi_turn_base_154",
    "bfcl::multi_turn_base_155",
    "bfcl::multi_turn_base_171",
    "bfcl::multi_turn_base_184",
    "bfcl::multi_turn_base_186",
    "bfcl::multi_turn_base_38",
    "bfcl::multi_turn_base_68",
    "bfcl::multi_turn_base_69",
    "bfcl::multi_turn_base_76",
    "bfcl::multi_turn_base_85",
    "bfcl::multi_turn_base_95",
    "bfcl::multi_turn_long_context_105",
    "bfcl::multi_turn_long_context_110",
    "bfcl::multi_turn_long_context_154",
    "bfcl::multi_turn_long_context_156",
    "bfcl::multi_turn_long_context_184",
    "bfcl::multi_turn_miss_func_154",
    "bfcl::multi_turn_miss_func_32",
    "bfcl::multi_turn_miss_func_44",
    "bfcl::multi_turn_miss_param_0",
    "bfcl::multi_turn_miss_param_1",
    "bfcl::multi_turn_miss_param_100",
    "bfcl::multi_turn_miss_param_109",
    "bfcl::multi_turn_miss_param_118",
    "bfcl::multi_turn_miss_param_131",
    "bfcl::multi_turn_miss_param_133",
    "bfcl::multi_turn_miss_param_136",
    "bfcl::multi_turn_miss_param_15",
    "bfcl::multi_turn_miss_param_152",
    "bfcl::multi_turn_miss_param_159",
    "bfcl::multi_turn_miss_param_169",
    "bfcl::multi_turn_miss_param_17",
    "bfcl::multi_turn_miss_param_176",
    "bfcl::multi_turn_miss_param_179",
    "bfcl::multi_turn_miss_param_18",
    "bfcl::multi_turn_miss_param_192",
    "bfcl::multi_turn_miss_param_28",
    "bfcl::multi_turn_miss_param_3",
    "bfcl::multi_turn_miss_param_36",
    "bfcl::multi_turn_miss_param_38",
    "bfcl::multi_turn_miss_param_39",
    "bfcl::multi_turn_miss_param_51",
    "bfcl::multi_turn_miss_param_73",
    "bfcl::multi_turn_miss_param_76",
    "bfcl::multi_turn_miss_param_96",
    "retail::11",
    "retail::21",
    "retail::27",
    "retail::39",
    "retail::64",
    "retail::68",
    "retail::81",
    "telecom::[mms_issue]bad_wifi_calling|break_apn_mms_setting|break_app_both_permissions|data_mode_off|data_usage_exceeded|user_abroad_roaming_disabled_off[PERSONA:None]",
    "telecom::[mobile_data_issue]bad_network_preference|bad_vpn|user_abroad_roaming_enabled_off[PERSONA:Easy]",
    "telecom::[mobile_data_issue]data_saver_mode_on|data_usage_exceeded[PERSONA:Easy]",
    "telecom::[mobile_data_issue]data_usage_exceeded|user_abroad_roaming_enabled_off[PERSONA:Easy]",
}
if DATASET == "final":  # the final dataset carries its split in every row (D-022)
    train_idx = np.array([i for i, r in enumerate(rows) if r["split"] == "train"])
    val_idx = np.array([i for i, r in enumerate(rows) if r["split"] == "val"])
else:
    assert VAL_GROUPS <= set(groups), "frozen validation groups that are not in the data: the data changed"
    val_idx = np.array([i for i, g in enumerate(groups) if g in VAL_GROUPS])
    train_idx = np.array([i for i, g in enumerate(groups) if g not in VAL_GROUPS])
# --- frozen split (end) ---
train_rows = [rows[i] for i in train_idx]
val_rows = [rows[i] for i in val_idx]

# --- integrity checks ---
assert not ({groups[i] for i in train_idx} & {groups[i] for i in val_idx}), "a group straddles train and val"
split_of = {rows[i]["trajectory_id"]: "train" for i in train_idx} | {rows[i]["trajectory_id"]: "val" for i in val_idx}
pairs = [r for r in rows if r.get("paired_negative_id")]
missing = [r["trajectory_id"] for r in pairs if r["paired_negative_id"] not in split_of]
assert not missing, f"{len(missing)} controls point at a negative that is not in the dataset"
assert all(split_of[r["trajectory_id"]] == split_of[r["paired_negative_id"]] for r in pairs), "a control pair was split"
by_tier = Counter(r["label_tier"] for r in pairs)
if DATASET == "legacy":
    assert set(by_tier) == {"synthetic_class2_control", "synthetic_class4_control", "synthetic_class6_control"}, by_tier
for cls in (2, 4, 6):
    ps = [r for r in pairs if r["label_tier"] == f"synthetic_class{cls}_control"]
    n_tr = sum(split_of[r["trajectory_id"]] == "train" for r in ps)
    print(f"Class {cls} pairs confirmed in the same split (via paired_negative_id): {len(ps)}  (train {n_tr}, val {len(ps) - n_tr})")
n_neg = {cls: sum(r["label"] == "incorrect" and r["error_class"] == cls for r in rows) for cls in (2, 4, 6)}
print("negatives without a control (no twin, so nothing to pair):", {f"class {c}": n_neg[c] - sum(r["label_tier"] == f"synthetic_class{c}_control" for r in pairs) for c in (2, 4, 6)})

for name, part in (("train", train_rows), ("val", val_rows)):
    n_inc = sum(r["label"] == "incorrect" for r in part)
    print(f"\n{name}: {len(part)} rows ({len(part) / len(rows):.1%})  incorrect={n_inc} ({n_inc / len(part):.1%})  correct={len(part) - n_inc}")
    print("  domains:", dict(Counter(r["domain"] for r in part)))
    print("  incorrect by class:", dict(sorted(Counter(r["error_class"] for r in part if r["label"] == "incorrect").items())))
print(f"\ngroups: train={len(set(groups[i] for i in train_idx))}  val={len(set(groups[i] for i in val_idx))}")
if DATASET == "legacy":
    assert (len(rows), len(train_rows), len(val_rows)) == (7580, 6816, 764), (len(rows), len(train_rows), len(val_rows))  # the frozen split, D-012
print(f"split sizes: train {len(train_rows)} / val {len(val_rows)}" + (" (documented 6,816 / 764)" if DATASET == "legacy" else " (taken from the split field of the rows)"))

if SMOKE:  # tiny subset so the whole notebook runs on a laptop CPU in a minute
    # keep real-origin rows in the smoke subset so that REAL_WEIGHT and the real-origin report are exercised
    train_rows = [r for r in train_rows if r.get("real_origin")][:48] + [r for r in train_rows if not r.get("real_origin")][:144]
    val_rows = [r for r in val_rows if r.get("real_origin")][:16] + [r for r in val_rows if not r.get("real_origin")][:48]
    print("SMOKE: using", len(train_rows), "train /", len(val_rows), "val rows")

## Cell 4: model and training

`answerdotai/ModernBERT-base` with `classifier_pooling="cls"` (the library default is mean pooling; this notebook uses the `[CLS]` token as specified) and SDPA attention. The sequence-classification head is ModernBERT's standard one: a small dense + GELU + LayerNorm block followed by the linear 2-way layer.

AdamW, lr 2e-5, warmup 10%, 3 epochs, effective batch 16, `max_length=4096`. Speed (D-020): training batches are length-bucketed (`LengthBucketSampler`: rows of similar length share a batch and the batches are shuffled; it prints the padding waste before and after), validation batches are length-sorted, and gradient checkpointing is off unless the GPU runs out of memory: the micro-batch ladder is 4, then 2 without it, then 2 and 1 with it (`GRADIENT_CHECKPOINTING = True/False` forces one side). The effective batch is always 16. `BENCHMARK_STEPS = 100` trains 100 steps, prints seconds per step and the projected time of a full run, and stops without saving. Best epoch by validation F1 (positive class = `incorrect`). Inputs are built with `format_input_v2`; checkpoints go to `checkpoints/phase5_v2/`.


In [ ]:
import math
import torch
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding, ModernBertConfig,
                          ModernBertForSequenceClassification, Trainer, TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint

LR, EPOCHS, WARMUP_RATIO, EFFECTIVE_BATCH = 2e-5, (1 if SMOKE else 3), 0.1, 16
TRAINER_DIR = CKPT_DIR / "trainer"
from transformers import TrainerCallback


class RunProgress(TrainerCallback):
    """Prints elapsed time and an estimated finish at every log line and every epoch, and stops training (after saving a
    checkpoint) once the time budget since the start of the notebook is used up."""

    def __init__(self):
        self.stopped_for_time = False

    def on_train_begin(self, args, state, control, **kw):
        self.t_start, self.t_epoch, self.step0 = time.time(), time.time(), state.global_step
        print(f"training starts at step {state.global_step}/{state.max_steps}; time budget {TIME_BUDGET_HOURS}h from notebook start")

    def _eta(self, state):
        done = state.global_step - self.step0
        if done <= 0:
            return None
        return (time.time() - self.t_start) / done * (state.max_steps - state.global_step)

    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            eta = self._eta(state)
            tail = "" if eta is None else f" | est. {fmt_td(eta)} left, finish ~{time.strftime('%H:%M', time.localtime(time.time() + eta))}"
            print(f"step {state.global_step}/{state.max_steps} loss {logs['loss']:.4f} | elapsed {fmt_td(time.time() - T0)}{tail}", flush=True)

    def on_epoch_end(self, args, state, control, **kw):
        eta = self._eta(state)
        tail = "" if eta is None else f", est. finish {time.strftime('%H:%M', time.localtime(time.time() + eta))} ({fmt_td(eta)} left)"
        print(f"=== epoch {state.epoch:.0f} finished in {fmt_td(time.time() - self.t_epoch)} | total elapsed {fmt_td(time.time() - T0)}{tail}", flush=True)
        self.t_epoch = time.time()

    def on_step_end(self, args, state, control, **kw):
        if not self.stopped_for_time and time.time() - T0 > TIME_BUDGET_HOURS * 3600:
            self.stopped_for_time = True
            control.should_save = True
            control.should_training_stop = True
            print(f"!!! time budget of {TIME_BUDGET_HOURS}h reached at step {state.global_step}/{state.max_steps}: saving a checkpoint and stopping. "
                  "Attach this run's output as an input of a new run to continue.", flush=True)


def training_complete(model_dir):
    """A saved model counts as finished unless its train_info.json says the run stopped early (time budget)."""
    if not ((model_dir / "config.json").exists() or (model_dir / "adapter_config.json").exists()):
        return False
    info = model_dir / "train_info.json"
    return json.load(open(info)).get("completed", True) if info.exists() else True


def micro_batch_plan(per_device, n_gpu):
    """accumulation steps that keep the effective batch at EFFECTIVE_BATCH: per-device batch x GPUs x accumulation."""
    n = max(n_gpu, 1)
    assert EFFECTIVE_BATCH % (per_device * n) == 0, f"{per_device} x {n} GPUs does not divide the effective batch {EFFECTIVE_BATCH}"
    return EFFECTIVE_BATCH // (per_device * n)

# --- length bucketing (D-020) ---
import random as _random


def padded_tokens(lengths, batches, multiple=8):
    """Tokens the model processes for these batches: every row is padded to the longest row of its batch (rounded up to `multiple`)."""
    total = 0
    for b in batches:
        longest = max(lengths[i] for i in b)
        total += len(b) * (-(-longest // multiple) * multiple)
    return total


class LengthBucketSampler:
    """transformers 5.x removed `group_by_length`, so this does it. Each epoch: a fresh random permutation is cut into chunks of
    `batch_size * mega` rows, each chunk is sorted by length and cut into batches, and then ALL batches are shuffled, so a batch holds
    rows of similar length and the order between batches is random. The batch with the longest row goes first: an out-of-memory error
    shows at step 1, and every epoch has the same worst-case batch shape. Yields row indices; every run of `batch_size` consecutive
    indices is one batch (a smaller leftover batch is last)."""

    def __init__(self, lengths, batch_size, seed=42, mega=50):
        self.lengths, self.batch_size, self.seed, self.mega, self.epoch = list(lengths), batch_size, seed, mega, 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def batches(self, epoch=None):
        epoch = self.epoch if epoch is None else epoch
        rng = _random.Random(self.seed * 1000003 + epoch)
        idx = list(range(len(self.lengths)))
        rng.shuffle(idx)
        chunk, bs, out = self.batch_size * self.mega, self.batch_size, []
        for s in range(0, len(idx), chunk):
            part = sorted(idx[s:s + chunk], key=lambda i: -self.lengths[i])
            out += [part[j:j + bs] for j in range(0, len(part), bs)]
        full, rest = [b for b in out if len(b) == bs], [b for b in out if len(b) < bs]
        rng.shuffle(full)
        if full:
            k = max(range(len(full)), key=lambda j: max(self.lengths[i] for i in full[j]))
            full.insert(0, full.pop(k))
        rest = [i for b in rest for i in b]
        return full + ([rest] if rest else [])  # the leftover rows (fewer than one batch) form the last batch

    def __iter__(self):
        for b in self.batches(self.epoch):
            yield from b
        self.epoch += 1

    def __len__(self):
        return len(self.lengths)


class LengthSortedSampler:
    """Evaluation order: shortest rows first, so every eval batch has rows of about the same length (the metrics do not depend on the order)."""

    def __init__(self, lengths):
        self.order = sorted(range(len(lengths)), key=lambda i: lengths[i])

    def __iter__(self):
        return iter(self.order)

    def __len__(self):
        return len(self.order)


import statistics
import time
from transformers import TrainerCallback


def row_lengths(ds):
    """Token count of every row of a tokenized dataset (a plain dataset object with .input_ids, or a Hugging Face Dataset)."""
    ids = ds["input_ids"] if hasattr(ds, "column_names") else ds.input_ids
    return [len(x) for x in ids]


def padding_report(lengths, batch):
    """Padding waste of the training set at this loader batch size: random order (what the Trainer did before) against length-bucketed."""
    real, rnd = sum(lengths), 0
    for s in range(3):
        o = list(range(len(lengths)))
        _random.Random(s).shuffle(o)
        rnd += padded_tokens(lengths, [o[i:i + batch] for i in range(0, len(o), batch)]) / 3
    buck = padded_tokens(lengths, LengthBucketSampler(lengths, batch).batches(0))
    print(f"padding waste at loader batch {batch}: random order {1 - real / rnd:.1%} ({rnd:,.0f} tokens processed for {real:,} real) -> "
          f"length-bucketed {1 - real / buck:.1%} ({buck:,.0f}); an epoch now processes {buck / rnd:.0%} of the tokens it did before")


class BenchmarkCallback(TrainerCallback):
    """BENCHMARK_STEPS mode: times every optimizer step (data loading included) and stops training after `n` of them."""

    def __init__(self, n):
        self.n, self.times, self.t0 = n, [], None

    def on_step_begin(self, args, state, control, **kw):
        self.t0 = time.perf_counter()

    def on_step_end(self, args, state, control, **kw):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.times.append(time.perf_counter() - self.t0)
        if len(self.times) >= self.n:
            control.should_training_stop = True


class LengthBucketTrainer(Trainer):
    """Trainer with length-bucketed training batches and length-sorted evaluation batches (D-020). It overrides two hooks that exist in
    transformers 4.x and 5.x and does not use `group_by_length` (its name and place changed between versions)."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.bench = None
        if BENCHMARK_STEPS:
            self.bench = BenchmarkCallback(BENCHMARK_STEPS)
            self.add_callback(self.bench)

    def _get_train_sampler(self, *args, **kwargs):
        ds = args[0] if args and args[0] is not None else self.train_dataset
        return LengthBucketSampler(row_lengths(ds), self._train_batch_size, seed=self.args.seed)

    def _get_eval_sampler(self, eval_dataset, *args, **kwargs):
        return LengthSortedSampler(row_lengths(eval_dataset))


def best_eval_metrics(trainer):
    """Validation metrics of the epoch that load_best_model_at_end restored, read from the per-epoch evaluations (no extra pass)."""
    step = getattr(trainer.state, "best_global_step", None)
    for h in trainer.state.log_history:
        if "eval_f1" in h and step and h.get("step") == step:
            return {k: v for k, v in h.items() if k.startswith("eval_")}
    print("best epoch not found in the log history: evaluating once more")
    return {k: v for k, v in trainer.evaluate().items() if k.startswith("eval_")}


def benchmark_report(trainer, n_gpu, batch_size, accum, ckpt):
    """BENCHMARK_STEPS mode: seconds per step, the projected time of a full run, then stop without saving anything."""
    t = trainer.bench.times
    skip = min(2, len(t) - 1)  # step 1 holds the longest batch and the CUDA warm-up
    use = t[skip:]
    mean, se = statistics.fmean(use), (statistics.pstdev(use) / len(use) ** 0.5 if len(use) > 1 else 0.0)
    t0 = time.perf_counter()
    trainer.evaluate()
    eval_s = time.perf_counter() - t0
    steps = math.ceil(len(train_ds) / EFFECTIVE_BATCH) * EPOCHS
    saves = EPOCHS * 90  # writing a checkpoint with optimizer state takes about a minute or two
    total = mean * steps + eval_s * EPOCHS + saves
    peak = f"{torch.cuda.max_memory_allocated() / 2**30:.1f} GiB peak GPU memory (one device)" if torch.cuda.is_available() else "no GPU"
    print("\n" + "=" * 100)
    print(f"BENCHMARK: {len(t)} optimizer steps | {n_gpu} GPU(s) x micro-batch {batch_size} x accumulation {accum} = {EFFECTIVE_BATCH} | gradient checkpointing {'on' if ckpt else 'off'} | {peak}")
    print(f"  step 1 (longest batch, warm-up): {t[0]:.1f}s | steps {skip + 1}-{len(t)}: mean {mean:.2f}s +- {se:.2f}s per step, min {min(use):.2f}s, max {max(use):.2f}s")
    print(f"  {EFFECTIVE_BATCH / mean:.2f} rows/s | validation pass {eval_s:.0f}s (length-sorted)")
    print(f"  full run = {steps} steps ({len(train_ds)} rows x {EPOCHS} epochs): {steps} x {mean:.2f}s + {EPOCHS} x {eval_s:.0f}s validation + about {saves // 60} min of saves "
          f"= {total / 3600:.2f} h ({int(total):,} s)")
    budget = globals().get("TIME_BUDGET_HOURS")
    if budget:
        print(f"  that is {'inside' if total / 3600 < budget else 'OUTSIDE'} the {budget} h time budget of this notebook")
    print("=" * 100)
    raise SystemExit("BENCHMARK_STEPS mode: stopped after the benchmark, nothing was saved. Set BENCHMARK_STEPS = None for the real run.")

# (per-device micro-batch, gradient checkpointing), fastest first; every entry gives an effective batch of 16. Gradient checkpointing recomputes the
# forward pass inside the backward pass (about 30% more compute) to save memory, so it is only used after an out-of-memory error without it. The longest
# batch comes first in every epoch (LengthBucketSampler), so a memory problem shows at step 1, not after hours.
LADDER = {None: [(4, False), (2, False), (2, True), (1, True)], True: [(2, True), (1, True)], False: [(4, False), (2, False)]}[GRADIENT_CHECKPOINTING]
EVAL_BATCH = 8  # validation is length-sorted and needs no gradients, so a larger batch costs little memory
# --- end length bucketing ---


class VerifierDataset(Dataset):
    def __init__(self, part):
        self.labels = [LABEL2ID[r["label"]] for r in part]
        enc = tokenizer([format_input_v2(r) for r in part], truncation=True, max_length=MAX_LEN)  # padding is dynamic, per batch
        self.input_ids, self.attention_mask = enc["input_ids"], enc["attention_mask"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "attention_mask": self.attention_mask[i], "labels": self.labels[i]}


def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(p.label_ids, preds), "precision": prec, "recall": rec, "f1": f1,
            "auroc": roc_auc_score(p.label_ids, probs)}


def make_model():
    set_seed(SEED)
    if SMOKE:  # randomly initialised toy model: exercises the code path only
        cfg = ModernBertConfig(vocab_size=len(tokenizer), hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                               num_attention_heads=2, num_labels=2, classifier_pooling="cls", pad_token_id=tokenizer.pad_token_id,
                               cls_token_id=tokenizer.cls_token_id, sep_token_id=tokenizer.sep_token_id, id2label=ID2LABEL, label2id=LABEL2ID)
        return ModernBertForSequenceClassification(cfg)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, classifier_pooling="cls", id2label=ID2LABEL, label2id=LABEL2ID, attn_implementation="sdpa")
    if hasattr(model.config, "reference_compile"):  # transformers 4.5x only: no torch.compile of the encoder (safer under DataParallel, unattended)
        model.config.reference_compile = False
    return model


if EVAL_ONLY:
    train_ds = val_ds = None
    print("EVAL_ONLY: training data is not tokenized")
else:
    assert int(REAL_WEIGHT) == REAL_WEIGHT and REAL_WEIGHT >= 1, "REAL_WEIGHT must be an integer >= 1"
    n_real_train = sum(bool(r.get("real_origin")) for r in train_rows)
    train_rows_w = train_rows + [r for r in train_rows if r.get("real_origin")] * (int(REAL_WEIGHT) - 1)
    print(f"REAL_WEIGHT = {REAL_WEIGHT}: {n_real_train} real-origin training rows of {len(train_rows)}; each appears {int(REAL_WEIGHT)} time(s) per epoch -> {len(train_rows_w)} rows per epoch"
          + (" (no change)" if REAL_WEIGHT == 1 else ""))
    train_ds, val_ds = VerifierDataset(train_rows_w), VerifierDataset(val_rows)
    print("train/val examples:", len(train_ds), len(val_ds), "| longest tokenized input:", max(len(x) for x in train_ds.input_ids))


def build_trainer(batch_size, n_gpu, callbacks, ckpt):
    accum = micro_batch_plan(batch_size, n_gpu)
    steps = math.ceil(len(train_ds) / (batch_size * max(n_gpu, 1) * accum)) * EPOCHS
    args = TrainingArguments(
        output_dir=str(TRAINER_DIR), num_train_epochs=EPOCHS, learning_rate=LR, optim="adamw_torch",
        lr_scheduler_type="linear", warmup_steps=int(WARMUP_RATIO * steps), weight_decay=0.01,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=EVAL_BATCH, gradient_accumulation_steps=accum,
        eval_strategy=("no" if BENCHMARK_STEPS else "epoch"), save_strategy=("no" if BENCHMARK_STEPS else "epoch"), save_total_limit=2,
        load_best_model_at_end=not BENCHMARK_STEPS, metric_for_best_model="f1", greater_is_better=True,
        gradient_checkpointing=ckpt, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=50, fp16=torch.cuda.is_available(), report_to="none", seed=SEED,
        disable_tqdm=(ENV == "kaggle"))  # progress lines come from RunProgress instead of a progress bar
    if torch.cuda.is_available() and n_gpu == 1 and args.n_gpu > 1:
        args._n_gpu = 1  # several GPUs are visible but this attempt must use GPU 0 only (no nn.DataParallel)
    return LengthBucketTrainer(model=make_model(), args=args, train_dataset=train_ds, eval_dataset=val_ds,
                   processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
                   compute_metrics=compute_metrics, callbacks=callbacks)

N_VISIBLE = torch.cuda.device_count()
N_GPU = N_VISIBLE if (USE_BOTH_GPUS and N_VISIBLE > 1) else min(N_VISIBLE, 1)
print(f"GPUs visible: {N_VISIBLE}; training will use {max(N_GPU, 1)} (nn.DataParallel when 2: per-device batch x GPUs x accumulation = {EFFECTIVE_BATCH})")

TRAINING_COMPLETE = False
if EVAL_ONLY:
    print("EVAL_ONLY = True: skipping training")
else:
    prior_best = restore_prior_run(CKPT_DIR)
    if prior_best is not None:
        BEST_DIR, TRAINING_COMPLETE = prior_best, True
    elif training_complete(BEST_DIR):
        print(f"Found trained model at {BEST_DIR}; skipping training.")
        TRAINING_COMPLETE = True
    else:
        resume = get_last_checkpoint(str(TRAINER_DIR)) if TRAINER_DIR.exists() else None
        print("resuming from", resume) if resume else print("starting fresh")
        progress = RunProgress()
        trained = False
        for n_gpu in ([N_GPU, 1] if N_GPU > 1 else [1]):  # two GPUs first; on any non-memory error, one GPU
            for batch_size, ckpt in LADDER:  # (per-device micro-batch, gradient checkpointing); all combinations give an effective batch of 16
                accum = micro_batch_plan(batch_size, n_gpu)
                trainer = build_trainer(batch_size, n_gpu, [progress], ckpt)
                try:
                    print(f"\ntraining with {n_gpu} GPU(s) x per-device batch {batch_size} x grad_accum {accum} = {EFFECTIVE_BATCH}, gradient checkpointing {'on' if ckpt else 'off'}")
                    padding_report(row_lengths(train_ds), batch_size * max(n_gpu, 1))
                    trainer.train(resume_from_checkpoint=resume)
                    trained = True
                    break
                except torch.cuda.OutOfMemoryError:
                    print("CUDA out of memory; retrying with a smaller micro-batch")
                except Exception as e:
                    if n_gpu > 1:
                        print(f"!!! two-GPU training failed ({type(e).__name__}: {str(e)[:300]}); FALLING BACK TO ONE GPU")
                        break
                    raise
                del trainer
                gc.collect()
                torch.cuda.empty_cache()
            if trained:
                break
        else:
            raise RuntimeError("out of memory even at batch size 1; lower MAX_LEN and TOKEN_BUDGET_V2")
        if BENCHMARK_STEPS:
            benchmark_report(trainer, n_gpu, batch_size, accum, ckpt)  # prints, then stops: nothing is saved
        TRAINING_COMPLETE = not progress.stopped_for_time

        trainer.save_model(str(BEST_DIR))          # the best epoch by val F1 (load_best_model_at_end)
        tokenizer.save_pretrained(str(BEST_DIR))
        best = best_eval_metrics(trainer)
        (BEST_DIR / "train_info.json").write_text(json.dumps({
            "val_metrics": best, "lr": LR, "epochs": EPOCHS, "effective_batch": EFFECTIVE_BATCH, "micro_batch": batch_size,
            "grad_accum": accum, "gradient_checkpointing": ckpt, "n_gpu": n_gpu, "max_len": MAX_LEN, "token_budget": TOKEN_BUDGET_V2, "input_format": "v2", "seed": SEED,
            "n_train": len(train_ds), "n_train_unique": len(train_rows), "real_weight": REAL_WEIGHT, "n_real_origin_train": n_real_train, "n_val": len(val_ds), "completed": TRAINING_COMPLETE,
            "elapsed_seconds": round(time.time() - T0), "log_history": trainer.state.log_history}, indent=1, default=str))
        print("\nbest-epoch validation metrics:", {k: round(v, 4) for k, v in best.items() if isinstance(v, float)})
        print("saved model + tokenizer to", BEST_DIR)
        if not TRAINING_COMPLETE:
            print("TRAINING IS NOT COMPLETE (time budget). The saved model is the best epoch so far; attach this output to a new run to continue.")
        elif ENV == "kaggle" and TRAINER_DIR.exists():
            shutil.rmtree(TRAINER_DIR)  # optimizer states are about 1.8 GB per checkpoint; the finished model is in best/
            print("removed the Trainer checkpoints (training is complete)")


## Cell 5: evaluate (legacy: `test_real.jsonl`; final: the 363-row test set, the 374-row synthetic test set and the validation set)

Positive class = `incorrect`. "False-flag rate" = share of the *correct* rows the model flags as incorrect. Legacy test set (D-010): 25 incorrect rows, so per-class and per-domain numbers are very noisy, and the correct half is "suspicious-looking but fine" calls rather than ordinary correct traffic.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

def load_eval_tokenizer(model_dir):
    """The tokenizer saved with the checkpoint, or the base model's tokenizer when the checkpoint has none."""
    if any((model_dir / f).exists() for f in ("tokenizer.json", "tokenizer_config.json", "vocab.json")):
        return AutoTokenizer.from_pretrained(str(model_dir))
    print(f"no tokenizer files in {model_dir}: using the tokenizer of {MODEL_NAME}")
    return tokenizer


EVAL_DIR = Path(EVAL_CHECKPOINT) if (EVAL_ONLY and EVAL_CHECKPOINT) else BEST_DIR
assert EVAL_DIR.exists(), f"nothing to evaluate: {EVAL_DIR} does not exist"
REPORT_PATH = (KAGGLE_WORKING if ENV == "kaggle" else CKPT_DIR) / "eval_report.txt"
tee = Tee(REPORT_PATH)  # from here on everything printed is also written to eval_report.txt
print(f"evaluation report | environment {ENV} | checkpoint {EVAL_DIR} | {time.strftime('%Y-%m-%d %H:%M:%S')}")
if not training_complete(EVAL_DIR) and not EVAL_ONLY:
    print("NOTE: this model comes from an unfinished run")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("evaluating on", device, "(CPU is slow for 4,096-token inputs but gives the same scores up to float precision)" if device == "cpu" else "")
model = AutoModelForSequenceClassification.from_pretrained(str(EVAL_DIR))
eval_tok = load_eval_tokenizer(EVAL_DIR)
model.to(device).eval()

TEST_NAME = "test_real" if DATASET == "legacy" else "the final test set"
PRED_FILE = "test_real_predictions.jsonl" if DATASET == "legacy" else "test_predictions.jsonl"
test_rows = load_jsonl(DATA / "test_real.jsonl") if DATASET == "legacy" else FINAL["test"]
y_true = np.array([LABEL2ID[r["label"]] for r in test_rows])
test_texts = [format_input_v2(r) for r in test_rows]


@torch.no_grad()
def predict_probs(texts, batch_size=4):  # inputs run up to 4,096 tokens
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))  # similar lengths together
    out = np.zeros(len(texts))
    for s in range(0, len(order), batch_size):
        idx = order[s : s + batch_size]
        enc = eval_tok([texts[i] for i in idx], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(**enc).logits
        out[idx] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    return out


probs = predict_probs(test_texts)


def metrics_at(t, mask=None):
    m = np.ones(len(y_true), bool) if mask is None else mask
    yt, pr = y_true[m], (probs[m] >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
    p, r, f, _ = precision_recall_fscore_support(yt, pr, average="binary", pos_label=1, zero_division=0)
    return dict(acc=(tp + tn) / len(yt), precision=p, recall=r, f1=f, false_flag=fp / max(fp + tn, 1), tn=tn, fp=fp, fn=fn, tp=tp)


THRESH = 0.5
o = metrics_at(THRESH)
print(f"=== overall on {TEST_NAME} (n={len(y_true)}, {int(y_true.sum())} incorrect), threshold {THRESH} ===")
print(f"accuracy {o['acc']:.3f}  precision {o['precision']:.3f}  recall {o['recall']:.3f}  F1 {o['f1']:.3f}")


def bootstrap_auroc(y, p, n_boot=1000, seed=0):
    # 95% percentile interval over row resamples; resamples that contain only one class are skipped
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            vals.append(roc_auc_score(y[i], p[i]))
    return np.percentile(vals, [2.5, 97.5])


lo, hi = bootstrap_auroc(y_true, probs)
print(f"AUROC {roc_auc_score(y_true, probs):.3f}  95% CI {lo:.3f}-{hi:.3f}  (1,000 bootstrap resamples, seed 0)")
no7 = np.array([r["error_class"] != 7 for r in test_rows])  # correct rows have error_class None, so they stay in
lo7, hi7 = bootstrap_auroc(y_true[no7], probs[no7])
print(f"AUROC without Class 7 {roc_auc_score(y_true[no7], probs[no7]):.3f}  95% CI {lo7:.3f}-{hi7:.3f}  "
      f"(n={int(no7.sum())}, {int(y_true[no7].sum())} incorrect)")
print(f"(always predicting 'correct' would score accuracy {1 - y_true.mean():.3f})")

print("\n=== recall per error class (incorrect rows only) ===")
for c in range(1, 8):
    m = np.array([r["label"] == "incorrect" and r["error_class"] == c for r in test_rows])
    if m.sum() == 0:
        print(f"  class {c}: n=0 (no real examples)")
    else:
        print(f"  class {c}: recall {(probs[m] >= THRESH).mean():.2f}  (n={int(m.sum())})")

print("\n=== per domain ===")
for d in ("retail", "airline", "telecom"):
    m = np.array([r["domain"] == d for r in test_rows])
    dm = metrics_at(THRESH, m)
    print(f"  {d:8s} n={int(m.sum()):3d} (incorrect={int(y_true[m].sum()):2d})  accuracy {dm['acc']:.3f}  F1 {dm['f1']:.3f}")

print("\n=== confusion matrix (rows = truth, columns = prediction) ===")
print(f"                   pred correct   pred incorrect")
print(f"  true correct     {o['tn']:12d}   {o['fp']:14d}")
print(f"  true incorrect   {o['fn']:12d}   {o['tp']:14d}")

print("\n=== threshold sweep ===")
print("  thresh      F1   precision  recall  false-flag rate")
for t in (0.3, 0.5, 0.7):
    s = metrics_at(t)
    print(f"  {t:5.1f}   {s['f1']:7.3f}   {s['precision']:8.3f}  {s['recall']:6.3f}  {s['false_flag']:14.3f}")

if DATASET == "final":
    # ================= final-mode report (D-023): real test set, per class by origin, per domain, synthetic test, baselines =================
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_curve

    def _boot(y, p, n_boot=1000, seed=0):  # 95% percentile interval over row resamples (same method as before)
        rng = np.random.default_rng(seed)
        vals = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            if y[i].min() != y[i].max():
                vals.append(roc_auc_score(y[i], p[i]))
        return np.percentile(vals, [2.5, 97.5])

    def _recall_at_fpr(y, p, fpr_max):
        fpr, tpr, thr = roc_curve(y, p)
        k = np.flatnonzero(fpr <= fpr_max)[-1]
        return float(tpr[k]), float(thr[k])

    def _cell(flagged, n):  # fewer than 10 examples: do not show a rate
        return f"too few to read ({flagged}/{n})" if n < 10 else f"recall {flagged / n:.2f} ({flagged}/{n})"

    def _length(r):
        return float(np.log1p(len(json.dumps(r["turns"])) + len(json.dumps(r["candidate_tool_call"]))))

    def _src(r):
        s = str(r.get("source", ""))
        return "induced" if s.startswith("induced") else "critictool" if s == "critictool" else "bfcl" if r["domain"] == "bfcl" else "tau2"

    SRC_NAMES = ["tau2", "bfcl", "critictool", "induced"]
    base_tr = np.array([r["label"] == "incorrect" for r in FINAL["train"]]).astype(int)

    def _baselines(part, yy):
        out = {}
        for name, feat in (("length only", lambda r: [_length(r)]), ("source only", lambda r: [float(_src(r) == s) for s in SRC_NAMES])):
            clf = LogisticRegression(max_iter=1000).fit(np.array([feat(r) for r in FINAL["train"]]), base_tr)
            sc = clf.predict_proba(np.array([feat(r) for r in part]))[:, 1]
            out[name] = roc_auc_score(yy, sc) if yy.min() != yy.max() else float("nan")
        return out

    def _overall(title, part, yy, pr):
        print(f"\n=== {title}: {len(yy)} rows, {int(yy.sum())} incorrect ===")
        lo, hi = _boot(yy, pr)
        pred = pr >= THRESH
        tp, fp = int((pred & (yy == 1)).sum()), int((pred & (yy == 0)).sum())
        prec = tp / max(tp + fp, 1)
        rec = tp / max(int(yy.sum()), 1)
        print(f"AUROC {roc_auc_score(yy, pr):.3f}  95% CI {lo:.3f}-{hi:.3f}  (1,000 bootstrap resamples, seed 0)")
        print(f"at threshold {THRESH}: precision {prec:.3f}  recall {rec:.3f}  F1 {2 * prec * rec / max(prec + rec, 1e-9):.3f}  false-flag rate {fp / max(int((yy == 0).sum()), 1):.3f}")
        for f in (0.05, 0.10):
            r_, t_ = _recall_at_fpr(yy, pr, f)
            print(f"recall at {f:.0%} false-alarm rate: {r_:.3f} (threshold {t_:.3f})")
        b = _baselines(part, yy)
        print(f"baselines on the same rows (trained on the training split): length only AUROC {b['length only']:.3f}, source only AUROC {b['source only']:.3f}")

    _overall("REAL TEST SET (human-reviewed)", test_rows, y_true, probs)
    origins = sorted({r["origin"] for r in test_rows})
    print(f"\nrecall per error class by origin (threshold {THRESH}; a cell with fewer than 10 examples is not shown as a rate)")
    for c in range(1, 8):
        cells = []
        for o in origins:
            m = np.array([r["label"] == "incorrect" and r["error_class"] == c and r["origin"] == o for r in test_rows])
            cells.append(f"{o}: " + _cell(int((probs[m] >= THRESH).sum()), int(m.sum())))
        print(f"  class {c}: " + " | ".join(cells))
    # Class 7 = policy violation (docs/labeling_guide.md). MAIN NUMBER: overall Class 7 recall. The sub-reason split below is approximate: since 2026-11-15 the
    # reasons come from a heuristic pre-fill applied by rule (D-022), not from a human, so it is labeled that way.
    m7 = np.array([r["label"] == "incorrect" and r["error_class"] == 7 for r in test_rows])
    print(f"\nCLASS 7 (policy violation) RECALL, threshold {THRESH}: " + _cell(int((probs[m7] >= THRESH).sum()), int(m7.sum())) + "   <- the number to quote")
    print("  Class 7 recall by sub-reason (heuristic tags, approximate; a cell with fewer than 10 examples is not shown as a rate):")
    for src_name, src_title in (("human", "reason confirmed by a human"), ("heuristic", "heuristic tags, approximate"), ("generator", "reason set by the generator")):
        shown = False
        for reason in ("no_confirmation", "ineligible_state", "missing_step", "precondition_error", "parallel_write", "other", "unlabeled"):
            m = np.array([r["label"] == "incorrect" and r["error_class"] == 7 and r.get("class7_reason") == reason and r.get("class7_reason_source") == src_name for r in test_rows])
            if m.any():
                if not shown:
                    print(f"    {src_title}:")
                    shown = True
                print(f"      {reason:20s} " + _cell(int((probs[m] >= THRESH).sum()), int(m.sum())))
    print("\nper domain")
    for d in sorted({r["domain"] for r in test_rows}):
        m = np.array([r["domain"] == d for r in test_rows])
        ni, nc = int(y_true[m].sum()), int((1 - y_true[m]).sum())
        auc = f"AUROC {roc_auc_score(y_true[m], probs[m]):.3f}" if ni >= 10 and nc >= 10 else "AUROC too few to read"
        print(f"  {d:8s} n {int(m.sum()):4d} (incorrect {ni:3d}, correct {nc:3d})  {auc}  recall {(probs[m & (y_true == 1)] >= THRESH).mean() if ni else float('nan'):.2f}  false-flag {(probs[m & (y_true == 0)] >= THRESH).mean() if nc else float('nan'):.2f}")

    synth = FINAL["test_synth"]
    probs_s = predict_probs([format_input_v2(r) for r in synth])
    ys = np.array([LABEL2ID[r["label"]] for r in synth])
    print("\n" + "-" * 78 + "\nSYNTHETIC TEST SET (held-out synthetic Classes 2 and 6 with their paired correct examples; NOT real errors)\n" + "-" * 78)
    _overall("SYNTHETIC held-out test", synth, ys, probs_s)
    for c in (2, 6):
        m = np.array([r["label"] == "incorrect" and r["error_class"] == c for r in synth])
        print(f"  synthetic class {c}: " + _cell(int((probs_s[m] >= THRESH).sum()), int(m.sum())))

    for name, part, pr in (("test_predictions.jsonl", test_rows, probs), ("test_synth_predictions.jsonl", synth, probs_s)):
        with open(CKPT_DIR / name, "w") as f:
            for r, p in zip(part, pr):
                f.write(json.dumps({"trajectory_id": r["trajectory_id"], "label": r["label"], "error_class": r.get("error_class"), "origin": r.get("origin"),
                                    "source": r.get("source"), "domain": r["domain"], "split": r.get("split"), "p_incorrect": float(p)}) + "\n")
    print("\nsaved", CKPT_DIR / "test_predictions.jsonl", "and", CKPT_DIR / "test_synth_predictions.jsonl")

# validation predictions of this model (D-025), with source, origin and real_origin, and the validation AUROC overall and per source group (D-026): the real-origin
# rows (real errors and their paired correct rows) are reported separately from the synthetic and external ones
if DATASET == "final":
    val_eval_rows = FINAL["val"]
    val_probs = predict_probs([format_input_v2(r) for r in val_eval_rows])
    with open(CKPT_DIR / "val_predictions.jsonl", "w") as f:
        for r, p in zip(val_eval_rows, val_probs):
            f.write(json.dumps({"trajectory_id": r["trajectory_id"], "label": r["label"], "error_class": r.get("error_class"), "origin": r.get("origin"),
                                "source": r.get("source"), "real_origin": bool(r.get("real_origin")), "domain": r["domain"], "split": r.get("split"), "p_incorrect": float(p)}) + "\n")
    yv = np.array([LABEL2ID[r["label"]] for r in val_eval_rows])
    print("saved", CKPT_DIR / "val_predictions.jsonl", f"(validation AUROC of this model {roc_auc_score(yv, val_probs):.3f})")

    def _val_group(r):
        if r.get("real_origin"):
            return "real-origin: " + str(r.get("origin"))
        if str(r.get("source", "")) == "critictool":
            return "CriticTool"
        if r["domain"] == "bfcl":
            return "BFCL"
        return "tau2 domains: synthetic, state flips, gold"

    print("\nvalidation AUROC by source group (errors and their correct rows of the group; the AUROC of a group is not defined without both labels):")
    groups_v = {}
    for i, r in enumerate(val_eval_rows):
        groups_v.setdefault(_val_group(r), []).append(i)
    for gname in sorted(groups_v):
        ix = np.array(groups_v[gname])
        n_inc = int(yv[ix].sum())
        if 0 < n_inc < len(ix):
            lo_v, hi_v = bootstrap_auroc(yv[ix], val_probs[ix])
            print(f"   {gname:44s} rows {len(ix):4d} (incorrect {n_inc:3d}, correct {len(ix) - n_inc:3d})  AUROC {roc_auc_score(yv[ix], val_probs[ix]):.3f}  95% CI {lo_v:.3f}-{hi_v:.3f}")
        else:
            print(f"   {gname:44s} rows {len(ix):4d} (incorrect {n_inc:3d}): AUROC not defined")
    ri = np.array([i for i, r in enumerate(val_eval_rows) if r.get("real_origin")])
    if len(ri) and 0 < yv[ri].sum() < len(ri):
        lo_r, hi_r = bootstrap_auroc(yv[ri], val_probs[ri])
        print(f"REAL-ORIGIN VALIDATION AUROC: {roc_auc_score(yv[ri], val_probs[ri]):.3f}  95% CI {lo_r:.3f}-{hi_r:.3f}  ({len(ri)} rows, {int(yv[ri].sum())} errors; few rows, so the interval is wide)")
    else:
        print("REAL-ORIGIN VALIDATION AUROC: not defined (the validation split has no real-origin errors or no real-origin correct rows)")

# save per-example probabilities for Phase 7 (calibration / risk-coverage). Legacy mode only: in final mode the richer file (with origin and source) was
# already written above and this block used to overwrite it (found in D-025)
if DATASET == "legacy":
    with open(CKPT_DIR / PRED_FILE, "w") as f:
        for r, p in zip(test_rows, probs):
            f.write(json.dumps({"trajectory_id": r["trajectory_id"], "label": r["label"], "error_class": r["error_class"],
                                "domain": r["domain"], "p_incorrect": float(p)}) + "\n")
    print("\nsaved", CKPT_DIR / PRED_FILE)
print("report written to", REPORT_PATH)
tee.close()

## Experiment REAL_CV (D-025, default off)

Fine-tunes the evaluated checkpoint on the human-reviewed real rows with grouped cross-validation and prints out-of-fold scores next to the plain ones. Set `REAL_CV = True` in the configuration cell; with `EVAL_ONLY = True` and `EVAL_CHECKPOINT` pointing at an earlier run's `best/` folder it takes about 30 minutes and trains nothing from scratch.


In [ ]:
# ===== Experiment REAL_CV (D-025): fine-tune the checkpoint on the human-reviewed real rows, grouped cross-validation. Default off. =====
if not REAL_CV:
    print("REAL_CV = False: skipped. Set it to True (with EVAL_ONLY = True and EVAL_CHECKPOINT = the final_v1 best/ folder) to run it.")
else:
    from transformers import Trainer, TrainingArguments
    from collections import defaultdict as _dd
    t_cv = time.time()
    cv_rows = FINAL["test"][:60] if SMOKE else FINAL["test"]
    y_cv = np.array([LABEL2ID[r["label"]] for r in cv_rows])
    assert DATASET == "final", "REAL_CV needs DATASET = 'final' (the human-reviewed test rows)"

    # 1. folds: whole (domain, task_id) groups, seeded, balanced on the number of incorrect rows and rows
    by_group = _dd(list)
    for i, r in enumerate(cv_rows):
        by_group[(r["domain"], str(r["task_id"]))].append(i)
    keys = sorted(by_group)
    random.Random(SEED).shuffle(keys)
    keys.sort(key=lambda k: -sum(y_cv[i] for i in by_group[k]))
    fold_of = np.zeros(len(cv_rows), int)
    load_ = [[0, 0] for _ in range(REAL_CV_FOLDS)]
    for k in keys:
        f = min(range(REAL_CV_FOLDS), key=lambda j: (load_[j][0], load_[j][1], j))
        for i in by_group[k]:
            fold_of[i] = f
        load_[f][0] += int(sum(y_cv[i] for i in by_group[k]))
        load_[f][1] += len(by_group[k])
    print(f"REAL_CV: {len(cv_rows)} rows in {len(keys)} task groups, {REAL_CV_FOLDS} folds (incorrect, rows per fold): {[tuple(x) for x in load_]}")
    assert all(len({fold_of[i] for i in idx}) == 1 for idx in by_group.values()), "a task group straddles two folds"

    def cv_predict(m, texts, batch_size=4):
        m.eval()
        order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
        out = np.zeros(len(texts))
        with torch.no_grad():
            for s in range(0, len(order), batch_size):
                idx = order[s : s + batch_size]
                enc = eval_tok([texts[i] for i in idx], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                    logits = m(**enc).logits
                out[idx] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
        return out

    # 2. one fine-tuning per fold, always starting from the evaluated checkpoint
    oof = np.zeros(len(cv_rows))
    for f in range(REAL_CV_FOLDS):
        tr = [r for r, ff in zip(cv_rows, fold_of) if ff != f]
        te_idx = [i for i in range(len(cv_rows)) if fold_of[i] == f]
        if not te_idx:
            continue
        t_f = time.time()
        ds = VerifierDataset(tr)
        done = False
        for micro, ckpt in ((2, False), (1, True)):
            ft = AutoModelForSequenceClassification.from_pretrained(str(EVAL_DIR))
            steps = math.ceil(len(ds) / EFFECTIVE_BATCH) * REAL_CV_EPOCHS
            targs = TrainingArguments(output_dir=str(CKPT_DIR / "real_cv_tmp"), num_train_epochs=REAL_CV_EPOCHS, learning_rate=REAL_CV_LR, optim="adamw_torch",
                                      lr_scheduler_type="linear", warmup_steps=max(1, int(0.1 * steps)), weight_decay=0.01, per_device_train_batch_size=micro,
                                      gradient_accumulation_steps=max(1, EFFECTIVE_BATCH // micro), eval_strategy="no", save_strategy="no", logging_steps=10 ** 9,
                                      gradient_checkpointing=ckpt, gradient_checkpointing_kwargs={"use_reentrant": False}, fp16=torch.cuda.is_available(),
                                      report_to="none", seed=SEED, disable_tqdm=True)
            tr_ = Trainer(model=ft, args=targs, train_dataset=ds, processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8))
            try:
                tr_.train()
                done = True
                break
            except torch.cuda.OutOfMemoryError:
                print("CUDA out of memory in the fold fine-tuning; retrying with micro-batch 1 and gradient checkpointing")
                del tr_, ft
                gc.collect()
                torch.cuda.empty_cache()
        assert done, "fold fine-tuning failed even at micro-batch 1"
        ft.to(device)
        oof[te_idx] = cv_predict(ft, [format_input_v2(cv_rows[i]) for i in te_idx])
        a = roc_auc_score(y_cv[te_idx], oof[te_idx]) if len(set(y_cv[te_idx])) == 2 else float("nan")
        print(f"   fold {f}: trained on {len(tr)} rows, scored {len(te_idx)} held-out rows, fold AUROC {a:.3f}, {time.time() - t_f:.0f}s")
        del tr_, ft, ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 3. the report: pooled out-of-fold scores against the plain scores of the same checkpoint on the same rows
    base = probs[: len(cv_rows)] if SMOKE else probs
    lo_b, hi_b = bootstrap_auroc(y_cv, base)
    lo_o, hi_o = bootstrap_auroc(y_cv, oof)
    print("\n" + "=" * 100)
    print(f"REAL_CV RESULT ({REAL_CV_FOLDS} folds of whole task groups, fine-tuning {REAL_CV_EPOCHS} epochs at lr {REAL_CV_LR}, from {EVAL_DIR})")
    print(f"  plain checkpoint, same {len(cv_rows)} rows      : AUROC {roc_auc_score(y_cv, base):.3f}  95% CI {lo_b:.3f}-{hi_b:.3f}")
    print(f"  fine-tuned on other folds (out-of-fold): AUROC {roc_auc_score(y_cv, oof):.3f}  95% CI {lo_o:.3f}-{hi_o:.3f}")
    rng = np.random.default_rng(0)
    dd = []
    for _ in range(1000):
        i = rng.integers(0, len(y_cv), len(y_cv))
        if y_cv[i].min() != y_cv[i].max():
            dd.append(roc_auc_score(y_cv[i], oof[i]) - roc_auc_score(y_cv[i], base[i]))
    print(f"  difference (out-of-fold - plain): {roc_auc_score(y_cv, oof) - roc_auc_score(y_cv, base):+.3f}  paired bootstrap 95% interval {np.percentile(dd, 2.5):+.3f} to {np.percentile(dd, 97.5):+.3f}")
    for fr in (0.05, 0.10):
        for nm, sc in (("plain", base), ("out-of-fold", oof)):
            thr = np.quantile(sc[y_cv == 0], 1 - fr)
            print(f"  recall at {fr:.0%} false alarms, {nm:11s}: {(sc[y_cv == 1] > thr).mean():.3f} (threshold {thr:.3f})")
    # the test rows carry a length artifact (the correct examples come from before the first error), so the fine-tuned model could win by length alone: score the same
    # folds with shortcut baselines fit on the training folds only (length; length + domain)
    from sklearn.linear_model import LogisticRegression
    Lg = np.array([[math.log1p(len(json.dumps(r["turns"])) + len(json.dumps(r["candidate_tool_call"])))] for r in cv_rows])
    Dm = np.array([[float(r["domain"] == d) for d in ("airline", "retail", "telecom")] for r in cv_rows])
    for nm, X in (("length only", Lg), ("length + domain", np.hstack([Lg, Dm]))):
        sc = np.zeros(len(cv_rows))
        for f in range(REAL_CV_FOLDS):
            te_i, tr_i = fold_of == f, fold_of != f
            if te_i.any() and len(set(y_cv[tr_i])) == 2:
                sc[te_i] = LogisticRegression(max_iter=1000).fit(X[tr_i], y_cv[tr_i]).predict_proba(X[te_i])[:, 1]
        print(f"  shortcut baseline on the same folds, {nm:16s}: out-of-fold AUROC {roc_auc_score(y_cv, sc):.3f}  <- the fine-tuned model has to beat this, not just the plain checkpoint")
    print("  recall at 0.5 per class and origin, plain -> out-of-fold (cells under 10 rows are not shown as rates):")
    for c in range(1, 8):
        cells = []
        for o in sorted({r.get("origin") for r in cv_rows}):
            m = np.array([r["label"] == "incorrect" and r.get("error_class") == c and r.get("origin") == o for r in cv_rows])
            if m.sum():
                cells.append(f"{o}: " + (f"{(base[m] >= 0.5).mean():.2f} -> {(oof[m] >= 0.5).mean():.2f} (n={int(m.sum())})" if m.sum() >= 10 else f"too few to read ({int((base[m] >= 0.5).sum())} -> {int((oof[m] >= 0.5).sum())} of {int(m.sum())})"))
        if cells:
            print(f"    class {c}: " + " | ".join(cells))
    for d in sorted({r["domain"] for r in cv_rows}):
        m = np.array([r["domain"] == d for r in cv_rows])
        if len(set(y_cv[m])) == 2:
            print(f"  {d:8s} n {int(m.sum()):4d}: AUROC plain {roc_auc_score(y_cv[m], base[m]):.3f} -> out-of-fold {roc_auc_score(y_cv[m], oof[m]):.3f}")
    with open(CKPT_DIR / "real_cv_predictions.jsonl", "w") as fh:
        for r, p, b, ff in zip(cv_rows, oof, base, fold_of):
            fh.write(json.dumps({"trajectory_id": r["trajectory_id"], "label": r["label"], "error_class": r.get("error_class"), "origin": r.get("origin"), "domain": r["domain"],
                                 "fold": int(ff), "p_incorrect_oof": float(p), "p_incorrect_plain": float(b)}) + "\n")
    print(f"saved {CKPT_DIR / 'real_cv_predictions.jsonl'}; REAL_CV took {fmt_td(time.time() - t_cv)}")
    shutil.rmtree(CKPT_DIR / "real_cv_tmp", ignore_errors=True)
